# Full experiment notebook (as run on Colab)

This is the complete notebook I ran while developing the project: dataset download, leak-free split, Task 1 (SAM/OWLv2 + review), Task 2 experiments T0-T5 including the failed ones, Task 3 evaluation and Task 4 video tracking including v1 and v2.
Outputs are cleared to keep the file small. The clean, final pipeline is in `run_all_colab.ipynb` and the `scripts/` folder. Add `ROBOFLOW_API_KEY` in Colab Secrets before running.

In [ ]:
!pip -q install roboflow opencv-python-headless imagehash scikit-image scikit-learn pandas matplotlib scipy tqdm yt-dlp tabulate
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

import os, glob, json, hashlib, shutil, random, math, warnings
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
PROJ = "/content/nosh_egg_project"
for d in ["images", "splits", "task1/labels", "task1/overlays", "task2", "task3", "task4"]:
    os.makedirs(f"{PROJ}/{d}", exist_ok=True)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
print("OpenCV:", cv2.__version__)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

# clean out the version-1 download and flattened copies
shutil.rmtree("/content/egg_raw", ignore_errors=True)
shutil.rmtree(f"{PROJ}/images", ignore_errors=True)
os.makedirs(f"{PROJ}/images", exist_ok=True)

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
rf.workspace("fire-bf7qe").project("egg-vzhtr").version(2).download("folder", location="/content/egg_raw")

raw = sorted(glob.glob("/content/egg_raw/**/*.jpg", recursive=True)
             + glob.glob("/content/egg_raw/**/*.jpeg", recursive=True)
             + glob.glob("/content/egg_raw/**/*.png", recursive=True))
print("raw images found:", len(raw))
assert len(raw) > 0, "No images found. Check the download output above."

# Flatten into one folder. Roboflow's own train/valid/test split is ignored; we make our own.
# Folder export layout: egg_raw/<split>/<class>/<image>
rows = []
for p in raw:
    parts = p.split("/")
    name = os.path.basename(p)
    dst = f"{PROJ}/images/{name}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
    with open(dst, "rb") as f:
        sha = hashlib.sha256(f.read()).hexdigest()
    h, w = cv2.imread(dst).shape[:2]
    rows.append(dict(path=dst, name=name, stem=os.path.splitext(name)[0], h=h, w=w,
                     sha256=sha, original_split=parts[-3], roboflow_class=parts[-2]))

manifest = pd.DataFrame(rows)
print("exact duplicate files:", manifest.sha256.duplicated().sum())
print("filename collisions (same name, different content):",
      (manifest.groupby("name").sha256.nunique() > 1).sum())
manifest = manifest.drop_duplicates("sha256").reset_index(drop=True)
print("images after exact dedup:", len(manifest))
print(manifest.original_split.value_counts())
print(manifest.roboflow_class.value_counts())
print("image sizes:", manifest[["h", "w"]].drop_duplicates().values.tolist()[:5])
print("files actually in images folder:", len(os.listdir(f"{PROJ}/images")))

In [ ]:
import imagehash
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

paths = manifest.path.tolist()
ph = [imagehash.phash(Image.open(p).convert("RGB")) for p in paths]
n = len(paths)

# pairwise Hamming distance on 64-bit pHash
D = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(i + 1, n):
        D[i, j] = D[j, i] = ph[i] - ph[j]
np.fill_diagonal(D, 99)
nn = D.min(axis=1)
print("nearest-neighbour distance histogram (distance: how many images):")
print(pd.Series(nn).value_counts().sort_index().head(15).to_string())

HASH_T = 8   # images within this distance are treated as the same scene
src = [os.path.basename(p).split(".rf.")[0] for p in paths]   # Roboflow parent name

parent = list(range(n))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for i in range(n):
    for j in range(i + 1, n):
        if D[i, j] <= HASH_T or src[i] == src[j]:
            parent[find(i)] = find(j)

manifest["group"] = [find(i) for i in range(n)]
sizes = manifest.group.value_counts()
print(f"\nHASH_T={HASH_T} | groups: {sizes.size} | largest group: {sizes.iloc[0]} | groups with >1 image: {(sizes > 1).sum()}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
dev_i, eval_i = next(gss.split(manifest, groups=manifest.group))
manifest["split"] = "dev"
manifest.loc[eval_i, "split"] = "eval"

dev_g, eval_g = set(manifest[manifest.split == "dev"].group), set(manifest[manifest.split == "eval"].group)
assert dev_g.isdisjoint(eval_g), "LEAKAGE: a group appears in both splits"
print("\nNo group appears in both dev and eval.")
print(manifest.split.value_counts())
print("dev:eval ratio = %.2f : 1" % (sum(manifest.split == "dev") / sum(manifest.split == "eval")))

manifest.to_csv(f"{PROJ}/splits/split_manifest.csv", index=False)

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

CKPT = "/content/sam_vit_h_4b8939.pth"
if not os.path.exists(CKPT):
    !wget -q -O {CKPT} https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
print("checkpoint size (GB):", round(os.path.getsize(CKPT) / 1e9, 2))   # expect about 2.56

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_h"](checkpoint=CKPT).to(device)

mask_gen = SamAutomaticMaskGenerator(
    sam, points_per_side=32, pred_iou_thresh=0.85,
    stability_score_thresh=0.88, min_mask_region_area=150)
predictor = SamPredictor(sam)
print("SAM loaded on", device)

# quick smoke test on one image, timing included
import time
test_path = manifest.path.iloc[0]
img = cv2.cvtColor(cv2.imread(test_path), cv2.COLOR_BGR2RGB)
t0 = time.time()
masks = mask_gen.generate(img)
print(f"smoke test: {len(masks)} raw SAM masks in {time.time() - t0:.1f}s")

In [ ]:
MIN_F, MAX_F = 0.004, 0.55   # egg area as a fraction of the image

def egg_candidates(img_rgb, masks, relaxed=False):
    H, W = img_rgb.shape[:2]
    ycc = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)
    skin = ((ycc[..., 1] > 135) & (ycc[..., 1] < 175) & (ycc[..., 2] > 85) & (ycc[..., 2] < 125))
    ell_thr, sol_thr, ar_thr = (0.75, 0.85, 2.6) if relaxed else (0.85, 0.92, 2.2)
    out = []
    for m in masks:
        seg = m["segmentation"]; frac = seg.sum() / (H * W)
        if not (MIN_F <= frac <= MAX_F): continue
        cnts, _ = cv2.findContours(seg.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        c = max(cnts, key=cv2.contourArea)
        if len(c) < 5: continue
        solidity = cv2.contourArea(c) / max(cv2.contourArea(cv2.convexHull(c)), 1)
        (cx, cy), (d1, d2), ang = cv2.fitEllipse(c)
        if min(d1, d2) < 5: continue
        ar = max(d1, d2) / min(d1, d2)
        em = np.zeros((H, W), np.uint8); cv2.ellipse(em, ((cx, cy), (d1, d2), ang), 1, -1); em = em.astype(bool)
        ell_iou = (seg & em).sum() / max((seg | em).sum(), 1)
        ys, xs = np.where(seg)
        touches = ys.min() == 0 or xs.min() == 0 or ys.max() == H - 1 or xs.max() == W - 1
        need = ell_thr - (0.08 if touches else 0)          # cropped eggs are less elliptical
        if ell_iou < need or solidity < sol_thr - (0.04 if touches else 0) or ar > ar_thr: continue
        skin_frac = skin[seg].mean()
        score = m["predicted_iou"] * m["stability_score"] * ell_iou * (1 - 0.5 * skin_frac)
        out.append(dict(seg=seg, score=float(score), ell_iou=float(ell_iou), solidity=float(solidity), skin=float(skin_frac)))
    return out

def nms_instances(cands, iou_thr=0.3, contain_thr=0.6):
    cands = sorted(cands, key=lambda d: -d["score"]); kept = []
    for c in cands:
        ok = True
        for k in kept:
            inter = (c["seg"] & k["seg"]).sum()
            iou = inter / max((c["seg"] | k["seg"]).sum(), 1)
            contain = inter / max(min(c["seg"].sum(), k["seg"].sum()), 1)
            if iou > iou_thr or contain > contain_thr: ok = False; break
        if ok: kept.append(c)
    return kept

def to_label_map(kept, shape):
    lbl = np.zeros(shape, np.int32)
    for i, k in enumerate(kept, 1): lbl[k["seg"] & (lbl == 0)] = i
    return lbl

PALETTE = [tuple(int(x) for x in cv2.cvtColor(np.uint8([[[(i * 47) % 180, 220, 255]]]), cv2.COLOR_HSV2RGB)[0, 0]) for i in range(64)]
def overlay(img_rgb, lbl, alpha=0.5):
    out = img_rgb.copy()
    for i in np.unique(lbl):
        if i == 0: continue
        m = lbl == i
        out[m] = (alpha * np.array(PALETTE[i % 64]) + (1 - alpha) * out[m]).astype(np.uint8)
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cs, -1, (255, 255, 255), 1)
    return out

def sam_egg_labels(img_rgb):
    masks = mask_gen.generate(img_rgb)
    kept = nms_instances(egg_candidates(img_rgb, masks)); mode = "strict"
    if not kept:
        kept = nms_instances(egg_candidates(img_rgb, masks, relaxed=True)); mode = "relaxed"
    return to_label_map(kept, img_rgb.shape[:2]), kept, mode, len(masks)

# ---- preview on 8 random DEV images only ----
prev = manifest[manifest.split == "dev"].sample(8, random_state=1)
fig, axs = plt.subplots(2, 8, figsize=(32, 8))
for c, r in enumerate(prev.itertuples()):
    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    lbl, kept, mode, nraw = sam_egg_labels(img)
    axs[0, c].imshow(img); axs[0, c].axis("off")
    axs[1, c].imshow(overlay(img, lbl)); axs[1, c].axis("off")
    axs[1, c].set_title(f"{len(kept)} eggs | {mode} | raw={nraw}", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
!pip -q install transformers
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from torchvision.ops import nms
from PIL import Image

OWL_NAME = "google/owlv2-base-patch16-ensemble"
owl_proc = Owlv2Processor.from_pretrained(OWL_NAME)
owl = Owlv2ForObjectDetection.from_pretrained(OWL_NAME).to(device).eval()
QUERIES = ["a photo of an egg"]

@torch.no_grad()
def owl_boxes(img_rgb):
    H, W = img_rgb.shape[:2]
    inputs = owl_proc(text=[QUERIES], images=Image.fromarray(img_rgb), return_tensors="pt").to(device)
    out = owl(**inputs)
    scores = torch.sigmoid(out.logits[0]).max(-1).values            # one score per candidate box
    b = out.pred_boxes[0]                                           # cx, cy, w, h relative to padded square
    S = max(H, W)
    xyxy = torch.stack([(b[:, 0] - b[:, 2] / 2) * S, (b[:, 1] - b[:, 3] / 2) * S,
                        (b[:, 0] + b[:, 2] / 2) * S, (b[:, 1] + b[:, 3] / 2) * S], 1)
    xyxy[:, 0::2] = xyxy[:, 0::2].clamp(0, W)
    xyxy[:, 1::2] = xyxy[:, 1::2].clamp(0, H)
    return xyxy.float().cpu().numpy(), scores.float().cpu().numpy()

def egg_labels_owl(img_rgb, abs_thr=0.10, rel_thr=0.4, nms_thr=0.5):
    H, W = img_rgb.shape[:2]
    boxes, scores = owl_boxes(img_rgb)
    top = float(scores.max())
    keep = scores >= max(abs_thr, rel_thr * top)        # absolute floor + relative to the best box in this image
    boxes, scores = boxes[keep], scores[keep]
    if len(boxes):
        idx = nms(torch.tensor(boxes), torch.tensor(scores), nms_thr).numpy()
        boxes, scores = boxes[idx], scores[idx]
    predictor.set_image(img_rgb)
    kept = []
    for b, s in zip(boxes, scores):
        m, q, _ = predictor.predict(box=b[None, :], multimask_output=False)
        seg = m[0]
        if seg.sum() < 0.3 * max((b[2] - b[0]) * (b[3] - b[1]), 1):
            continue                                    # SAM mask barely fills its box: reject
        kept.append(dict(seg=seg, score=float(s) * float(q[0]), det=float(s), box=b))
    kept = nms_instances(kept, iou_thr=0.5, contain_thr=0.8)
    return to_label_map(kept, (H, W)), kept, top

# same 8 DEV images as before, so you can compare directly
prev = manifest[manifest.split == "dev"].sample(8, random_state=1)
fig, axs = plt.subplots(2, 8, figsize=(32, 8))
for c, r in enumerate(prev.itertuples()):
    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    lbl, kept, top = egg_labels_owl(img)
    boxed = img.copy()
    for k in kept:
        x0, y0, x1, y1 = [int(v) for v in k["box"]]
        cv2.rectangle(boxed, (x0, y0), (x1, y1), (255, 0, 0), 3)
        cv2.putText(boxed, f"{k['det']:.2f}", (x0 + 5, y0 + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    axs[0, c].imshow(boxed); axs[0, c].axis("off")
    axs[1, c].imshow(overlay(img, lbl)); axs[1, c].axis("off")
    axs[1, c].set_title(f"{len(kept)} eggs | best det score {top:.2f}", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
from scipy import ndimage as ndi

QUERIES = ["a photo of an egg", "a white egg",
           "a partially visible egg cut off at the edge of the image"]

def clean_mask(seg):
    seg = ndi.binary_fill_holes(seg)
    n, cc, stats, _ = cv2.connectedComponentsWithStats(seg.astype(np.uint8), 8)
    if n > 2:
        seg = cc == (1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA])))
    return seg

def egg_labels_owl(img_rgb):
    H, W = img_rgb.shape[:2]
    boxes_all, scores_all = owl_boxes(img_rgb)          # uses the new QUERIES list
    top = float(scores_all.max())
    predictor.set_image(img_rgb)

    def run(abs_thr, rel_thr, nms_thr=0.5):
        keep = scores_all >= max(abs_thr, rel_thr * top)
        b, s = boxes_all[keep], scores_all[keep]
        if len(b):
            idx = nms(torch.tensor(b), torch.tensor(s), nms_thr).numpy()
            b, s = b[idx], s[idx]
        kept = []
        for bb, ss in zip(b, s):
            m, q, _ = predictor.predict(box=bb[None, :], multimask_output=False)
            seg = clean_mask(m[0])
            if seg.sum() < 0.3 * max((bb[2] - bb[0]) * (bb[3] - bb[1]), 1):
                continue
            kept.append(dict(seg=seg, score=float(ss) * float(q[0]), det=float(ss), box=bb))
        return nms_instances(kept, iou_thr=0.5, contain_thr=0.8)

    kept = run(0.10, 0.4); status = "strict"
    if not kept:
        kept = run(0.03, 0.5); status = "fallback"
    if not kept:
        status = "none"
    return to_label_map(kept, (H, W)), kept, top, status

# ---- run on all 160 images ----
rows = []
for r in tqdm(manifest.itertuples(), total=len(manifest)):
    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    lbl, kept, top, status = egg_labels_owl(img)
    np.save(f"{PROJ}/task1/labels/{r.stem}.npy", lbl)
    cv2.imwrite(f"{PROJ}/task1/overlays/{r.stem}.jpg", cv2.cvtColor(overlay(img, lbl), cv2.COLOR_RGB2BGR))
    min_det = min([k["det"] for k in kept], default=0.0)
    rows.append(dict(stem=r.stem, split=r.split, n_eggs=len(kept), status=status, top=round(top, 3),
                     min_det=round(min_det, 3),
                     needs_review=(status != "strict" or len(kept) == 0 or len(kept) >= 4 or min_det < 0.15)))
t1 = pd.DataFrame(rows)
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)

print(t1.status.value_counts().to_string(), "\n")
print("eggs per image:\n", t1.n_eggs.value_counts().sort_index().to_string(), "\n")
print("flagged for review:", int(t1.needs_review.sum()), "of", len(t1))

# ---- sheet of flagged images, worst first ----
flag = t1[t1.needs_review].sort_values(["n_eggs", "top"]).head(16)
ncols = 4; nrows = max(1, math.ceil(len(flag) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows)); axs = np.array(axs).reshape(-1)
for a in axs: a.axis("off")
for a, r in zip(axs, flag.itertuples()):
    a.imshow(cv2.cvtColor(cv2.imread(f"{PROJ}/task1/overlays/{r.stem}.jpg"), cv2.COLOR_BGR2RGB))
    a.set_title(f"{r.stem[:22]}\n{r.status} n={r.n_eggs} top={r.top}", fontsize=9)
plt.tight_layout(); plt.show()

# back up Task 1 results to Drive (your GPU quota can disappear mid-session)
bk = "/content/drive/MyDrive/Nosh_Egg_Project_Backup"
os.makedirs(bk, exist_ok=True)
shutil.copytree(f"{PROJ}/task1", f"{bk}/task1", dirs_exist_ok=True)
shutil.copytree(f"{PROJ}/splits", f"{bk}/splits", dirs_exist_ok=True)
print("backed up to Drive:", bk)

In [ ]:
def _path(stem): return manifest.loc[manifest.stem == stem, "path"].iloc[0]

def get_candidates(img_rgb, max_n=8):
    masks = mask_gen.generate(img_rgb)
    H, W = img_rgb.shape[:2]
    c = []
    for m in masks:
        seg = m["segmentation"]; fr = seg.sum() / (H * W)
        if fr < 0.03 or fr > 0.92: continue
        c.append(dict(seg=seg, score=m["predicted_iou"] * m["stability_score"]))
    c = nms_instances(c, iou_thr=0.6, contain_thr=0.9)      # sorts by score, drops near-duplicates
    return c[:max_n]

def label_point(seg):
    d = cv2.distanceTransform(seg.astype(np.uint8), cv2.DIST_L2, 5)
    y, x = np.unravel_index(np.argmax(d), d.shape)
    return int(x), int(y)

COLORS = [(255,0,0),(0,170,0),(0,0,255),(255,140,0),(160,0,200),(0,170,170),(200,200,0),(255,0,140)]
def draw_cands(img_rgb, cands):
    out = img_rgb.copy()
    for k, c in enumerate(cands):
        cs, _ = cv2.findContours(c["seg"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cs, -1, COLORS[k % len(COLORS)], 3)
    for k, c in enumerate(cands):
        x, y = label_point(c["seg"])
        cv2.putText(out, str(k + 1), (x - 12, y + 12), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 0, 0), 6)
        cv2.putText(out, str(k + 1), (x - 12, y + 12), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (255, 255, 255), 3)
    return out

none_stems = t1[t1.status == "none"].stem.tolist()
cand_store = {}
for s in tqdm(none_stems):
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
    cand_store[s] = get_candidates(img)

for b0 in range(0, len(none_stems), 6):
    batch = list(range(b0, min(b0 + 6, len(none_stems))))
    fig, axs = plt.subplots(2, 3, figsize=(21, 14)); axs = axs.reshape(-1)
    for a in axs: a.axis("off")
    for a, i in zip(axs, batch):
        s = none_stems[i]
        img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
        a.imshow(draw_cands(img, cand_store[s]))
        a.set_title(f"#{i}   ({len(cand_store[s])} candidates)", fontsize=18)
    plt.tight_layout(); plt.show()

In [ ]:
CHOICES = {0: [2], 1: [1], 2: [], 3: [1], 4: [1], 5: [1], 6: [],
           7: [1], 8: [3], 9: [1], 10: [2], 11: [2], 12: [2], 13: [1]}

for i, picks in CHOICES.items():
    s = none_stems[i]
    cands = cand_store[s]
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
    lbl = np.zeros(img.shape[:2], np.int32)
    for j, k in enumerate(picks, 1):
        seg = clean_mask(cands[k - 1]["seg"])
        lbl[seg & (lbl == 0)] = j
    np.save(f"{PROJ}/task1/labels/{s}.npy", lbl)
    cv2.imwrite(f"{PROJ}/task1/overlays/{s}.jpg", cv2.cvtColor(overlay(img, lbl), cv2.COLOR_RGB2BGR))
    m = t1.stem == s
    t1.loc[m, "n_eggs"] = len(picks)
    t1.loc[m, "status"] = "manual_pick" if picks else "no_egg"
    t1.loc[m, "needs_review"] = False
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)
print(t1.status.value_counts().to_string())

# check the 14 results
fig, axs = plt.subplots(2, 7, figsize=(28, 8)); axs = axs.reshape(-1)
for a in axs: a.axis("off")
for a, i in zip(axs, range(len(none_stems))):
    s = none_stems[i]
    a.imshow(cv2.cvtColor(cv2.imread(f"{PROJ}/task1/overlays/{s}.jpg"), cv2.COLOR_BGR2RGB))
    a.set_title(f"#{i} {t1.loc[t1.stem == s, 'status'].iloc[0]}", fontsize=14)
plt.tight_layout(); plt.show()

shutil.copytree(f"{PROJ}/task1", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task1", dirs_exist_ok=True)
print("backed up")

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX
def overlay_numbered(img_rgb, lbl):
    out = overlay(img_rgb, lbl)
    for i in np.unique(lbl):
        if i == 0: continue
        x, y = label_point(lbl == i)
        cv2.putText(out, str(i), (x - 12, y + 12), FONT, 1.6, (0, 0, 0), 6)
        cv2.putText(out, str(i), (x - 12, y + 12), FONT, 1.6, (255, 255, 255), 3)
    return out

rev_stems  = t1[(t1.status == "fallback") | ((t1.status == "strict") & (t1.n_eggs >= 2))].stem.tolist()
spot_stems = t1[(t1.status == "strict") & (t1.n_eggs == 1)].sample(12, random_state=0).stem.tolist()
print("to review:", len(rev_stems), "| random strict sample:", len(spot_stems))

def show_group(stems, prefix, per=12):
    for b0 in range(0, len(stems), per):
        batch = range(b0, min(b0 + per, len(stems)))
        fig, axs = plt.subplots(3, 4, figsize=(20, 15)); axs = axs.reshape(-1)
        for a in axs: a.axis("off")
        for a, i in zip(axs, batch):
            s = stems[i]
            img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
            lbl = np.load(f"{PROJ}/task1/labels/{s}.npy")
            row = t1[t1.stem == s].iloc[0]
            a.imshow(overlay_numbered(img, lbl))
            a.set_title(f"{prefix}{i}   {row.status}   n={row.n_eggs}", fontsize=15)
        plt.tight_layout(); plt.show()

show_group(rev_stems, "R")
show_group(spot_stems, "S")

In [ ]:
def clean_relabel(lbl):
    out = np.zeros_like(lbl); k = 0
    for i in np.unique(lbl):
        if i == 0: continue
        k += 1; out[lbl == i] = k
    return out

def seg_from_points(stem, pos, neg=()):
    img = cv2.cvtColor(cv2.imread(_path(stem)), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    pts = np.array(list(pos) + list(neg)); lab = np.array([1] * len(pos) + [0] * len(neg))
    ms, sc, _ = predictor.predict(point_coords=pts, point_labels=lab, multimask_output=True)
    return clean_mask(ms[int(np.argmax(sc))])

REMOVE = {6: [1, 3], 14: [2], 15: [1], 21: [3], 24: [2]}     # R index: egg numbers to delete
ADD    = {5: [(610, 350)], 22: [(290, 40)]}                   # R index: (x, y) points on a missed egg

fix_ids = sorted(set(REMOVE) | set(ADD))
for i in fix_ids:
    s = rev_stems[i]; path = f"{PROJ}/task1/labels/{s}.npy"
    lbl = np.load(path)
    for k in REMOVE.get(i, []): lbl[lbl == k] = 0
    lbl = clean_relabel(lbl)
    for pt in ADD.get(i, []):
        seg = seg_from_points(s, [pt]) & (lbl == 0)
        lbl[seg] = lbl.max() + 1
    np.save(path, lbl)
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
    cv2.imwrite(f"{PROJ}/task1/overlays/{s}.jpg", cv2.cvtColor(overlay(img, lbl), cv2.COLOR_RGB2BGR))
    m = t1.stem == s
    t1.loc[m, "n_eggs"] = int(lbl.max()); t1.loc[m, "status"] = "manual_fix"; t1.loc[m, "needs_review"] = False
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)

fig, axs = plt.subplots(2, 4, figsize=(22, 11)); axs = axs.reshape(-1)
for a in axs: a.axis("off")
for a, i in zip(axs, fix_ids):
    s = rev_stems[i]
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
    a.imshow(overlay_numbered(img, np.load(f"{PROJ}/task1/labels/{s}.npy"))); a.set_title(f"R{i} (fixed)", fontsize=16)
plt.tight_layout(); plt.show()

shutil.copytree(f"{PROJ}/task1", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task1", dirs_exist_ok=True)
print(t1.status.value_counts().to_string())

In [ ]:
strict_stems = t1[(t1.status == "strict") & (t1.n_eggs == 1)].stem.tolist()
print("single-egg strict images:", len(strict_stems))

PER = 48
for b0 in range(0, len(strict_stems), PER):
    batch = list(range(b0, min(b0 + PER, len(strict_stems))))
    fig, axs = plt.subplots(6, 8, figsize=(32, 24)); axs = axs.reshape(-1)
    for a in axs: a.axis("off")
    for a, i in zip(axs, batch):
        s = strict_stems[i]
        a.imshow(cv2.cvtColor(cv2.imread(f"{PROJ}/task1/overlays/{s}.jpg"), cv2.COLOR_BGR2RGB))
        a.set_title(str(i), fontsize=22)
    plt.tight_layout(); plt.show()

In [ ]:
def propose_extra(img_rgb, lbl, abs_thr=0.05, rel_thr=0.10, min_area=0.012, max_area=0.7):
    H, W = img_rgb.shape[:2]
    boxes_all, scores_all = owl_boxes(img_rgb)
    top = float(scores_all.max())
    keep = scores_all >= max(abs_thr, rel_thr * top)
    b, s = boxes_all[keep], scores_all[keep]
    if len(b) == 0: return []
    idx = nms(torch.tensor(b), torch.tensor(s), 0.5).numpy(); b, s = b[idx], s[idx]
    predictor.set_image(img_rgb)
    existing = lbl > 0
    cands = []
    for bb, ss in zip(b, s):
        m, q, _ = predictor.predict(box=bb[None, :], multimask_output=False)
        seg = clean_mask(m[0]); a = seg.sum()
        if a < min_area * H * W or a > max_area * H * W: continue
        if (seg & existing).sum() / a > 0.2: continue          # already covered by an existing egg
        cands.append(dict(seg=seg, score=float(ss), det=float(ss)))
    return nms_instances(cands, iou_thr=0.5, contain_thr=0.8)

elig = t1[t1.status.isin(["strict", "fallback", "manual_fix"])]
proposals = {}
for r in tqdm(elig.itertuples(), total=len(elig)):
    # Use the _path() helper function to resolve the file path via stem
    img_path = _path(r.stem)
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    lbl = np.load(f"{PROJ}/task1/labels/{r.stem}.npy")
    pr = propose_extra(img, lbl)
    if pr: proposals[r.stem] = pr
prop_stems = list(proposals.keys())
print("images with proposals:", len(prop_stems), "| total proposals:", sum(len(v) for v in proposals.values()))

def draw_props(img_rgb, lbl, props):
    out = img_rgb.copy()
    for i in np.unique(lbl):
        if i == 0: continue
        cs, _ = cv2.findContours((lbl == i).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cs, -1, (0, 255, 255), 2)                     # existing eggs: cyan outline
    for p in props:
        ov = out.copy(); ov[p["seg"]] = (255, 0, 0); out = cv2.addWeighted(ov, 0.4, out, 0.6, 0)
        cs, _ = cv2.findContours(p["seg"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cs, -1, (255, 0, 0), 3)                       # proposals: red
    for k, p in enumerate(props, 1):
        x, y = label_point(p["seg"])
        cv2.putText(out, str(k), (x - 12, y + 12), FONT, 1.6, (0, 0, 0), 6)
        cv2.putText(out, str(k), (x - 12, y + 12), FONT, 1.6, (255, 255, 255), 3)
    return out

for b0 in range(0, len(prop_stems), 12):
    batch = list(range(b0, min(b0 + 12, len(prop_stems))))
    fig, axs = plt.subplots(3, 4, figsize=(20, 15)); axs = axs.reshape(-1)
    for a in axs: a.axis("off")
    for a, i in zip(axs, batch):
        s = prop_stems[i]
        img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
        lbl = np.load(f"{PROJ}/task1/labels/{s}.npy")
        a.imshow(draw_props(img, lbl, proposals[s]))
        a.set_title(f"P{i}   det: " + ", ".join(f"{p['det']:.2f}" for p in proposals[s]), fontsize=15)
    plt.tight_layout(); plt.show()

In [ ]:
ACCEPT       = {2: [1, 2], 4: [1, 2], 7: [1], 8: [1], 10: [1], 28: [1, 2]}   # P index: proposal numbers to accept
EXTRA_POINTS = {20: [(420, 335)], 101: [(180, 80)]}                           # strict_stems index: (x, y) on a missed egg

changed = []
for i, picks in ACCEPT.items():
    s = prop_stems[i]; path = f"{PROJ}/task1/labels/{s}.npy"
    lbl = np.load(path)
    for k in picks:
        seg = proposals[s][k - 1]["seg"] & (lbl == 0)
        lbl[seg] = lbl.max() + 1
    np.save(path, lbl); changed.append(s)

for i, pts in EXTRA_POINTS.items():
    s = strict_stems[i]; path = f"{PROJ}/task1/labels/{s}.npy"
    lbl = np.load(path)
    for pt in pts:
        seg = seg_from_points(s, [pt]) & (lbl == 0)
        lbl[seg] = lbl.max() + 1
    np.save(path, lbl); changed.append(s)

for s in changed:
    lbl = np.load(f"{PROJ}/task1/labels/{s}.npy")
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
    cv2.imwrite(f"{PROJ}/task1/overlays/{s}.jpg", cv2.cvtColor(overlay(img, lbl), cv2.COLOR_RGB2BGR))
    m = t1.stem == s
    t1.loc[m, "n_eggs"] = int(lbl.max()); t1.loc[m, "status"] = "manual_add"; t1.loc[m, "needs_review"] = False
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)

fig, axs = plt.subplots(2, 4, figsize=(22, 11)); axs = axs.reshape(-1)
for a in axs: a.axis("off")
for a, s in zip(axs, changed):
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
    a.imshow(overlay_numbered(img, np.load(f"{PROJ}/task1/labels/{s}.npy")))
    a.set_title(f"{t1.loc[t1.stem == s, 'n_eggs'].iloc[0]} eggs", fontsize=16)
plt.tight_layout(); plt.show()

shutil.copytree(f"{PROJ}/task1", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task1", dirs_exist_ok=True)
print(t1.status.value_counts().to_string())
print("total eggs in ground truth:", int(t1.n_eggs.sum()))

In [ ]:
def seg_from_box(stem, box):
    img = cv2.cvtColor(cv2.imread(_path(stem)), cv2.COLOR_BGR2RGB)
    predictor.set_image(img)
    ms, sc, _ = predictor.predict(box=np.array(box, dtype=np.float32)[None, :], multimask_output=True)
    return clean_mask(ms[int(np.argmax(sc))])

s = strict_stems[20]; path = f"{PROJ}/task1/labels/{s}.npy"
lbl = np.load(path)
seg = seg_from_box(s, [0, 222, 506, 619])
new = ((lbl == 1) | seg).astype(np.int32)            # keep the existing lower-left mask and add the rest of the egg
np.save(path, new)

img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
cv2.imwrite(f"{PROJ}/task1/overlays/{s}.jpg", cv2.cvtColor(overlay(img, new), cv2.COLOR_RGB2BGR))
m = t1.stem == s
t1.loc[m, "n_eggs"] = 1; t1.loc[m, "status"] = "manual_add"
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(14, 7))
axs[0].imshow(img); axs[0].set_title("original"); axs[0].axis("off")
axs[1].imshow(overlay(img, new)); axs[1].set_title("1 egg: box prompt + existing mask"); axs[1].axis("off")
plt.tight_layout(); plt.show()

shutil.copytree(f"{PROJ}/task1", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task1", dirs_exist_ok=True)
print("total eggs in ground truth:", int(t1.n_eggs.sum()))

In [ ]:
s = strict_stems[20]; path = f"{PROJ}/task1/labels/{s}.npy"
img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
predictor.set_image(img)
ms, sc, _ = predictor.predict(box=np.array([0, 222, 506, 619], dtype=np.float32)[None, :], multimask_output=True)
sam_opts = [ndi.binary_fill_holes(m) for m in ms]          # no largest-blob step: keep every piece of the egg

# colour option: white, low-saturation pixels inside the box (egg = white, leaves = green)
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
box_mask = np.zeros(img.shape[:2], bool); box_mask[222:619, 0:506] = True
white = (hsv[..., 1] < 50) & (hsv[..., 2] > 140) & box_mask
white = cv2.morphologyEx(white.astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8)) > 0
white = ndi.binary_fill_holes(white)
cnt, cc, st, _ = cv2.connectedComponentsWithStats(white.astype(np.uint8), 8)
keep = np.zeros_like(white)
for k in range(1, cnt):
    if st[k, cv2.CC_STAT_AREA] >= 400: keep |= cc == k      # keep every sizeable piece, not just the biggest
old = np.load(path) == 1
color_union = keep | old

opts = [("original", None)] + [(f"SAM mask {k} (score {sc[k]:.2f})", sam_opts[k]) for k in range(3)] \
       + [("white pixels + current mask", color_union)]
fig, axs = plt.subplots(1, 5, figsize=(30, 6))
for a, (t, m) in zip(axs, opts):
    a.imshow(img if m is None else overlay(img, m.astype(np.int32))); a.set_title(t, fontsize=14); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import shutil, glob

# 1) save SAM mask 0 (all pieces kept, holes filled) as the final mask for this image
s = strict_stems[20]
lbl_path = f"{PROJ}/task1/labels/{s}.npy"
final = sam_opts[0].astype(np.int32)
np.save(lbl_path, final)

img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB)
cv2.imwrite(f"{PROJ}/task1/overlays/{s}.jpg", cv2.cvtColor(overlay(img, final), cv2.COLOR_RGB2BGR))

t1.loc[t1.stem == s, ["n_eggs", "status"]] = [1, "manual_fix"]
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)

# 2) freeze Task 1: recount everything straight from the saved label files
rows = []
for f in sorted(glob.glob(f"{PROJ}/task1/labels/*.npy")):
    L = np.load(f)
    rows.append((os.path.basename(f)[:-4], int(L.max()), float((L > 0).mean())))
chk = pd.DataFrame(rows, columns=["stem", "n_eggs_file", "fg_frac"])
t1 = t1.drop(columns=[c for c in ["n_eggs_file", "fg_frac"] if c in t1.columns]).merge(chk, on="stem", how="left")
t1["n_eggs"] = t1["n_eggs_file"]            # the label files are the source of truth
t1.to_csv(f"{PROJ}/task1/task1_summary.csv", index=False)

print("images with labels :", len(chk), "of", len(manifest))
print("total GT eggs      :", int(t1.n_eggs.sum()))
print("images with 0 eggs :", int((t1.n_eggs == 0).sum()))
print("images with 2+ eggs:", int((t1.n_eggs >= 2).sum()))
print("\nby status:\n", t1.status.value_counts().to_string())
print("\nby split:\n", t1.groupby("split").agg(images=("stem", "count"), eggs=("n_eggs", "sum")).to_string())

# 3) back up to Drive
dst = "/content/drive/MyDrive/Nosh_Egg_Project_Backup"
for d in ["task1", "splits"]:
    shutil.copytree(f"{PROJ}/{d}", f"{dst}/{d}", dirs_exist_ok=True)
print("\nbacked up to", dst)

In [ ]:
from scipy.optimize import linear_sum_assignment
import time, json
os.makedirs(f"{PROJ}/task2", exist_ok=True)

dev_stems = t1[t1.split == "dev"].stem.tolist()
assert len(dev_stems) == 128
DEV = {s: (cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB), np.load(f"{PROJ}/task1/labels/{s}.npy")) for s in dev_stems}
print("dev images loaded:", len(DEV))

# ---------------- metrics ----------------
def match_instances(gt, pr):
    g, p = gt.max(), pr.max()
    if g == 0 or p == 0: return [], g, p
    pair = np.bincount((gt.ravel() * (p + 1) + pr.ravel()), minlength=(g + 1) * (p + 1)).reshape(g + 1, p + 1)
    inter = pair[1:, 1:].astype(float)
    ga = pair[1:, :].sum(1)
    pa = pair[:, 1:].sum(0)  # FIXED: Removed the extra [1:] slice which caused the shape mismatch
    iou = inter / (ga[:, None] + pa[None, :] - inter + 1e-9)
    r, c = linear_sum_assignment(-iou)
    return [(i, j, iou[i, j]) for i, j in zip(r, c)], g, p

def evaluate(seg_fn, stems, thr=0.5, return_per_image=False):
    TP = FP = FN = 0; ious = []; fg_iou = []; fg_dice = []; cnt_err = []; per = {}
    for s in stems:
        img, gt = DEV[s] if s in DEV else (None, None)
        pr = seg_fn(img)
        m, g, p = match_instances(gt, pr)
        tp = sum(1 for _, _, v in m if v >= thr)
        TP += tp; FP += p - tp; FN += g - tp
        ious += [v for _, _, v in m if v >= thr]
        a, b = gt > 0, pr > 0
        if a.any():
            fg_iou.append((a & b).sum() / ((a | b).sum() + 1e-9))
            fg_dice.append(2 * (a & b).sum() / (a.sum() + b.sum() + 1e-9))
        cnt_err.append(abs(int(g) - int(p)))
        per[s] = dict(gt=int(g), pred=int(p), tp=tp, fg_iou=(fg_iou[-1] if a.any() else np.nan))
    prec = TP / max(TP + FP, 1); rec = TP / max(TP + FN, 1)
    res = dict(precision=prec, recall=rec, F1_at_0_5=2 * prec * rec / max(prec + rec, 1e-9),
               mean_inst_IoU=float(np.mean(ious)) if ious else 0.0,
               fg_IoU=float(np.mean(fg_iou)), fg_Dice=float(np.mean(fg_dice)), count_MAE=float(np.mean(cnt_err)))
    return (res, per) if return_per_image else res

# ---------------- experiment log ----------------
LOG_PATH = f"{PROJ}/task2/experiment_log.csv"
exp_log = pd.read_csv(LOG_PATH).to_dict("records") if os.path.exists(LOG_PATH) else []

def run_trial(tid, desc, seg_fn, outcome_note=""):
    t0 = time.time(); res = evaluate(seg_fn, dev_stems)
    row = dict(trial=tid, description=desc, **{k: round(v, 4) for k, v in res.items()},
               sec=round(time.time() - t0, 1), note=outcome_note)
    exp_log[:] = [r for r in exp_log if r["trial"] != tid] + [row]
    pd.DataFrame(exp_log).to_csv(LOG_PATH, index=False)
    print(pd.DataFrame([row]).T.to_string(header=False)); return res

# ---------------- shared post-processing: binary mask -> instance label map ----------------
def binary_to_instances(b, min_area=500):
    b = ndi.binary_fill_holes(b).astype(np.uint8)
    n, cc, st, _ = cv2.connectedComponentsWithStats(b, 8)
    out = np.zeros(b.shape, np.int32); k = 0
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] >= min_area: k += 1; out[cc == i] = k
    return out

K5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# ---------------- T0: grayscale + global Otsu ----------------
def seg_T0(img):
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    b = cv2.morphologyEx(cv2.morphologyEx(b, cv2.MORPH_OPEN, K5), cv2.MORPH_CLOSE, K5) > 0
    return binary_to_instances(b)

# ---------------- T1: LAB "whiteness" = lightness - chroma, Otsu ----------------
def whiteness(img):
    lab = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_RGB2LAB).astype(np.float32)
    L, a, b = lab[..., 0], lab[..., 1] - 128, lab[..., 2] - 128
    w = L - 1.5 * np.sqrt(a ** 2 + b ** 2)
    return cv2.normalize(w, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def seg_T1(img):
    _, b = cv2.threshold(whiteness(img), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    b = cv2.morphologyEx(cv2.morphologyEx(b, cv2.MORPH_OPEN, K5), cv2.MORPH_CLOSE, K5) > 0
    return binary_to_instances(b)

r0 = run_trial("T0", "Gray + Gaussian + global Otsu + open/close + fill holes", seg_T0, "baseline")
r1 = run_trial("T1", "LAB whiteness (L - 1.5*chroma) + global Otsu + open/close", seg_T1, "colour cue instead of raw brightness")

# ---------------- look at the failures of T1 (GT contour = green, prediction = red) ----------------
_, per = evaluate(seg_T1, dev_stems, return_per_image=True)
worst = sorted(per, key=lambda s: (np.nan_to_num(per[s]["fg_iou"], nan=1.0)))[:8]
fig, axs = plt.subplots(2, 4, figsize=(22, 11))
for a, s in zip(axs.ravel(), worst):
    img, gt = DEV[s]; pr = seg_T1(img); vis = img.copy()
    for m_, col in [(gt > 0, (0, 255, 0)), (pr > 0, (255, 0, 0))]:
        cs, _ = cv2.findContours(m_.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, cs, -1, col, 3)
    a.imshow(vis); a.set_title(f"{s[:22]}  GT={per[s]['gt']} pred={per[s]['pred']} fgIoU={per[s]['fg_iou']:.2f}", fontsize=10); a.axis("off")
plt.suptitle("T1 worst dev images   green = ground truth, red = classical prediction", fontsize=16)
plt.tight_layout(); plt.show()

In [ ]:
# ---------------- 0a) BUG FIX in my Cell 15 scorer: the predicted-area vector dropped one entry ----------------
def match_instances(gt, pr):
    g, p = int(gt.max()), int(pr.max())
    if g == 0 or p == 0: return [], g, p
    pair = np.bincount((gt.ravel() * (p + 1) + pr.ravel()), minlength=(g + 1) * (p + 1)).reshape(g + 1, p + 1)
    inter = pair[1:, 1:].astype(float)
    ga, pa = pair[1:, :].sum(1), pair[:, 1:].sum(0)          # (was pair[:, 1:].sum(0)[1:], which lost one entry)
    iou = inter / (ga[:, None] + pa[None, :] - inter + 1e-9)
    r, c = linear_sum_assignment(-iou)
    return [(i, j, iou[i, j]) for i, j in zip(r, c)], g, p

# re-score the two earlier trials with the corrected scorer (overwrites their rows in the log)
run_trial("T0", "Gray + Gaussian + global Otsu + open/close + fill holes", seg_T0, "baseline (re-scored after scorer fix)")
run_trial("T1", "LAB whiteness (L - 1.5*chroma) + global Otsu + open/close", seg_T1, "colour cue instead of raw brightness (re-scored after scorer fix)")

# ---------------- 0b) evidence: are eggs brighter or darker than their surroundings? ----------------
rows = []
for s in dev_stems:
    img, gt = DEV[s]; fg = gt > 0
    if fg.any() and (~fg).any():
        L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[..., 0].astype(float)
        rows.append(dict(stem=s, L_in=L[fg].mean(), L_out=L[~fg].mean(), area=fg.mean(),
                         touches_border=bool(fg[0].any() or fg[-1].any() or fg[:, 0].any() or fg[:, -1].any())))
diag = pd.DataFrame(rows); diag["darker_than_bg"] = diag.L_in < diag.L_out
print(f"eggs DARKER than background: {diag.darker_than_bg.mean()*100:.0f}% of dev images")
print(f"egg touches image border   : {diag.touches_border.mean()*100:.0f}%")
print(f"egg area fraction  median {diag.area.median():.2f}  (10th-90th pct {diag.area.quantile(.1):.2f}-{diag.area.quantile(.9):.2f})")

annotate = lambda tid, note: [r.update(note=note) for r in exp_log if r["trial"] == tid] and pd.DataFrame(exp_log).to_csv(LOG_PATH, index=False)
annotate("T0", "FAIL: assumes egg is the bright region; many eggs are brown/dark and background is bright -> picks wall, cloth, hand")
annotate("T1", "FAIL (slightly better): same polarity assumption; whiteness picks bright wall/tray/hand instead of the egg")

# ---------------- T2: propose many candidate regions (both polarities, several channels), score egg-likeness, keep best ----------------
K9 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
MIN_A = 0.02

def feature_channels(img):
    blur = cv2.GaussianBlur(img, (7, 7), 0)
    lab = cv2.cvtColor(blur, cv2.COLOR_RGB2LAB); hsv = cv2.cvtColor(blur, cv2.COLOR_RGB2HSV)
    return dict(L=lab[..., 0], a=lab[..., 1], b=lab[..., 2], S=hsv[..., 1], W=whiteness(img))

def candidate_masks(img):
    H, W = img.shape[:2]; out = []
    for name, ch in feature_channels(img).items():
        _, m = cv2.threshold(ch, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        for pol, mm in (("hi", m), ("lo", 255 - m)):
            mm = cv2.morphologyEx(cv2.morphologyEx(mm, cv2.MORPH_OPEN, K9), cv2.MORPH_CLOSE, K9)
            n, cc, st, _ = cv2.connectedComponentsWithStats((mm > 0).astype(np.uint8), 8)
            for i in range(1, n):
                if st[i, cv2.CC_STAT_AREA] >= MIN_A * H * W:
                    out.append((f"{name}_{pol}", ndi.binary_fill_holes(cc == i)))
    return out

def score_component(comp, grad, lab):
    area = comp.mean()
    if area < MIN_A or area > 0.9: return -1.0
    cs, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cs, key=cv2.contourArea)
    sol = cv2.contourArea(c) / (cv2.contourArea(cv2.convexHull(c)) + 1e-9)          # convex, egg-like shape
    bd = comp & ~ndi.binary_erosion(comp, iterations=2)
    bd[:4, :] = bd[-4:, :] = False; bd[:, :4] = bd[:, -4:] = False                    # ignore the image border itself
    edge = grad[bd].mean() if bd.sum() > 30 else 0.0                                  # boundary sits on a real edge
    homog = 1.0 / (1.0 + lab[comp].std(0).mean() / 10.0)                              # smooth, uniform surface
    return 0.4 * sol + 0.4 * (edge / (edge + 15.0)) + 0.2 * homog

_CACHE = {}
def ranked_candidates(img, nms_iou=0.3):
    key = id(img)
    if key in _CACHE: return _CACHE[key]
    lab = cv2.cvtColor(cv2.GaussianBlur(img, (5, 5), 0), cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[..., 0]; grad = np.hypot(cv2.Sobel(L, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(L, cv2.CV_32F, 0, 1, ksize=3)) / 4.0
    cands = [(n, m, score_component(m, grad, lab)) for n, m in candidate_masks(img)]
    cands = sorted([c for c in cands if c[2] > 0], key=lambda c: -c[2]); kept = []
    for n, m, sc in cands:
        if all((m & k[1]).sum() / ((m | k[1]).sum() + 1e-9) < nms_iou for k in kept): kept.append((n, m, sc))
    _CACHE[key] = kept; return kept

def make_seg_T2(thr_add):
    def f(img):
        kept = ranked_candidates(img); out = np.zeros(img.shape[:2], np.int32); k = 0
        for j, (n, m, sc) in enumerate(kept):
            if j == 0 or sc >= thr_add:                      # always keep the best candidate, others only if convincing
                free = m & (out == 0)
                if free.sum() >= MIN_A * m.size: k += 1; out[free] = k
        return out
    return f

print("\nscoring candidates for all dev images (cached) ...")
t0 = time.time(); [ranked_candidates(DEV[s][0]) for s in dev_stems]; print(f"done in {time.time()-t0:.0f}s")
print("\nthr_add grid (dev only):")
grid = {}
for thr in [0.55, 0.65, 0.75, 0.85, 1.01]:
    grid[thr] = evaluate(make_seg_T2(thr), dev_stems)
    print(f"  thr_add={thr:.2f}  F1={grid[thr]['F1_at_0_5']:.3f}  P={grid[thr]['precision']:.3f}  R={grid[thr]['recall']:.3f}  fgIoU={grid[thr]['fg_IoU']:.3f}  MAE={grid[thr]['count_MAE']:.2f}")
best_thr = max(grid, key=lambda t: grid[t]["F1_at_0_5"])
seg_T2 = make_seg_T2(best_thr)
run_trial("T2", f"Multi-channel (L,a,b,S,whiteness) x both polarities Otsu -> candidates scored by convexity+edge+homogeneity, NMS, thr_add={best_thr}",
          seg_T2, "polarity-agnostic propose-and-rank")

def show_worst(seg_fn, title, k=8):
    _, per = evaluate(seg_fn, dev_stems, return_per_image=True)
    worst = sorted(per, key=lambda s: np.nan_to_num(per[s]["fg_iou"], nan=1.0))[:k]
    fig, axs = plt.subplots(2, 4, figsize=(22, 11))
    for a, s in zip(axs.ravel(), worst):
        img, gt = DEV[s]; pr = seg_fn(img); vis = img.copy()
        for m_, col in [(gt > 0, (0, 255, 0)), (pr > 0, (255, 0, 0))]:
            cs, _ = cv2.findContours(m_.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(vis, cs, -1, col, 3)
        a.imshow(vis); a.set_title(f"{s[:22]} GT={per[s]['gt']} pred={per[s]['pred']} fgIoU={per[s]['fg_iou']:.2f}", fontsize=10); a.axis("off")
    plt.suptitle(title + "   green = ground truth, red = classical prediction", fontsize=16); plt.tight_layout(); plt.show()

show_worst(seg_T2, "T2 worst dev images")

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib

S, R = 160, 320
CHN = [f"{c}_{p}" for c in ["L", "a", "b", "S", "W"] for p in ("hi", "lo")]
FEAT_NAMES = ["area", "solidity", "ellipse_fit", "border_frac", "edge", "contrast", "abs_contrast", "texture",
              "color_std", "center_dist", "mean_L", "chroma", "circularity", "consensus70", "consensus50"] + CHN
ds = lambda m, n=S: cv2.resize(m.astype(np.uint8), (n, n), interpolation=cv2.INTER_NEAREST).astype(bool)
pack = lambda m: np.packbits(m.ravel())
unpack = lambda p, shp: np.unpackbits(p)[:shp[0] * shp[1]].reshape(shp).astype(bool)
iou_b = lambda a, b: (a & b).sum() / ((a | b).sum() + 1e-9)

def cand_feats(comp_full, name, lab, grad):
    comp = ds(comp_full, R); H, W = comp.shape
    if comp.sum() < 50: return None
    cs, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cs, key=cv2.contourArea); ca = cv2.contourArea(c) + 1e-9; peri = cv2.arcLength(c, True)
    sol = ca / (cv2.contourArea(cv2.convexHull(c)) + 1e-9)
    ell = 0.0
    if len(c) >= 5:
        em = np.zeros((H, W), np.uint8); cv2.ellipse(em, cv2.fitEllipse(c), 1, -1); em = em.astype(bool); ell = iou_b(em, comp)
    bd = comp & ~ndi.binary_erosion(comp, iterations=1)
    onb = bd.copy(); onb[2:-2, 2:-2] = False; border_frac = onb.sum() / (bd.sum() + 1e-9)
    bdi = bd & ~onb; edge = grad[bdi].mean() if bdi.sum() > 15 else 0.0
    ring = ndi.binary_dilation(comp, iterations=6) & ~comp; L = lab[..., 0]
    contrast = L[comp].mean() - (L[ring].mean() if ring.any() else L[comp].mean())
    inner = ndi.binary_erosion(comp, iterations=3); inner = inner if inner.sum() > 30 else comp
    ys, xs = np.nonzero(comp); cdist = np.hypot(ys.mean() / H - .5, xs.mean() / W - .5)
    chroma = np.hypot(lab[..., 1] - 128, lab[..., 2] - 128)[comp].mean()
    return [comp.mean(), sol, ell, border_frac, edge, contrast, abs(contrast), grad[inner].mean(),
            lab[comp].std(0).mean(), cdist, L[comp].mean(), chroma, 4 * np.pi * ca / (peri ** 2 + 1e-9)]

print("building candidate pool + features for all dev images ...")
t0 = time.time(); DATA = {}; ID2STEM = {}
for s in dev_stems:
    img, gt = DEV[s]; ID2STEM[id(img)] = s
    im_r = cv2.resize(img, (R, R), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(cv2.GaussianBlur(im_r, (5, 5), 0), cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[..., 0]; grad = np.hypot(cv2.Sobel(L, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(L, cv2.CV_32F, 0, 1, ksize=3)) / 4.0
    feats, small, packed, names = [], [], [], []
    for name, m in candidate_masks(img):
        f = cand_feats(m, name, lab, grad)
        if f is None: continue
        feats.append(f + [0, 0] + [1.0 * (name == n) for n in CHN]); small.append(ds(m)); packed.append(pack(m)); names.append(name)
    if not feats: DATA[s] = dict(X=np.zeros((0, len(FEAT_NAMES))), y=np.zeros(0), small=[], packed=[], names=[], shape=img.shape[:2]); continue
    X = np.array(feats, float); sm = np.stack(small).reshape(len(small), -1).astype(np.float32)
    inter = sm @ sm.T; ar = sm.sum(1); iou = inter / (ar[:, None] + ar[None, :] - inter + 1e-9); np.fill_diagonal(iou, 0)
    X[:, 13] = (iou > 0.7).sum(1); X[:, 14] = (iou > 0.5).sum(1)                       # how many other proposals agree
    gts = [ds(gt == k) for k in range(1, int(gt.max()) + 1)]
    y = np.array([max([iou_b(m, g) for g in gts], default=0.0) for m in small])
    DATA[s] = dict(X=np.nan_to_num(X), y=y, small=small, packed=packed, names=names, shape=img.shape[:2], gts=gts)
print(f"done in {time.time()-t0:.0f}s;  candidates per image: median {np.median([len(DATA[s]['names']) for s in dev_stems]):.0f}")

# ---------------- ORACLE: does a good proposal exist at all? ----------------
best, who = [], []
for s in dev_stems:
    d = DATA[s]
    for g in d.get("gts", []):
        ious = [iou_b(m, g) for m in d["small"]]
        if ious: best.append(max(ious)); who.append(d["names"][int(np.argmax(ious))])
        else: best.append(0.0); who.append("none")
best = np.array(best)
print(f"\nORACLE over {len(best)} GT eggs (best proposal IoU, if we always picked the right one):")
for t in (0.5, 0.7, 0.8, 0.9): print(f"   IoU >= {t}: {np.mean(best >= t)*100:.0f}%")
print("   mean best IoU:", round(best.mean(), 3)); print("   which channel proposed the best region:\n", pd.Series(who).value_counts().head(8).to_string())

# ---------------- learned ranker (dev only, grouped cross-validation so near-duplicates never straddle folds) ----------------
grp = manifest.set_index("stem").loc[dev_stems, "group"].values
Xall = np.vstack([DATA[s]["X"] for s in dev_stems]); yall = np.concatenate([DATA[s]["y"] for s in dev_stems])
owner = np.concatenate([[i] * len(DATA[s]["y"]) for i, s in enumerate(dev_stems)])
def oof_scores(make_model, k=5):
    sc = {}
    for tr, te in GroupKFold(k).split(dev_stems, groups=grp):
        mdl = make_model().fit(Xall[np.isin(owner, tr)], yall[np.isin(owner, tr)])
        for i in te:
            X = DATA[dev_stems[i]]["X"]; sc[dev_stems[i]] = mdl.predict(X) if len(X) else np.zeros(0)
    return sc

def make_seg_learned(scores, thr):
    def f(img):
        s = ID2STEM[id(img)]; d = DATA[s]; sc = scores[s]; out = np.zeros(d["shape"], np.int32)
        kept, k = [], 0
        for j, i in enumerate(np.argsort(-sc)):
            if j > 0 and sc[i] < thr: break
            if all(iou_b(d["small"][i], d["small"][q]) < 0.3 for q in kept):
                kept.append(i); m = unpack(d["packed"][i], d["shape"]); free = m & (out == 0)
                if free.sum() >= MIN_A * m.size: k += 1; out[free] = k
        return out
    return f

models = {"T3": ("Ridge", lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
          "T4": ("RandomForest", lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=0))}
SEGS = {}
for tid, (nm, mk_) in models.items():
    sc = oof_scores(mk_); grid = {}
    for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        grid[thr] = evaluate(make_seg_learned(sc, thr), dev_stems)
    bt = max(grid, key=lambda t: grid[t]["F1_at_0_5"]); SEGS[tid] = make_seg_learned(sc, bt)
    print(f"\n{tid} {nm}: threshold grid -> " + "  ".join(f"{t}:{grid[t]['F1_at_0_5']:.3f}" for t in grid) + f"   (best {bt})")
    run_trial(tid, f"Learned proposal ranker ({nm} on 28 shape/edge/colour/consensus features, 5-fold grouped CV on dev), thr_add={bt}",
              SEGS[tid], "dev scores are out-of-fold")
    mdl = mk_().fit(Xall, yall); joblib.dump(dict(model=mdl, features=FEAT_NAMES, thr=bt), f"{PROJ}/task2/ranker_{nm}.joblib")
    if nm == "RandomForest":
        imp = pd.Series(mdl.feature_importances_, FEAT_NAMES).sort_values(ascending=False).head(8); print("\nRF top features:\n", imp.round(3).to_string())

bestT = max(["T3", "T4"], key=lambda t: [r for r in exp_log if r["trial"] == t][0]["F1_at_0_5"])
print("\nbest learned ranker:", bestT); show_worst(SEGS[bestT], f"{bestT} worst dev images")

In [ ]:
FEAT2 = FEAT_NAMES + ["refined"]; NF1 = len(FEAT_NAMES)
for r in exp_log:                                                      # fix the feature count in the T3/T4 descriptions
    if r["trial"] in ("T3", "T4"): r["description"] = r["description"].replace("28 shape", f"{NF1} shape")
pd.DataFrame(exp_log).to_csv(LOG_PATH, index=False)

# ---------------- reusable pipeline pieces (same code is used later for eval images and for the video) ----------------
def img_ctx(img):
    im_r = cv2.resize(img, (R, R), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(cv2.GaussianBlur(im_r, (5, 5), 0), cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[..., 0]; grad = np.hypot(cv2.Sobel(L, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(L, cv2.CV_32F, 0, 1, ksize=3)) / 4.0
    return im_r, lab, grad

def rows_for(masks, names, lab, grad, refined):
    rows, sm, pk, nm = [], [], [], []
    for m, n in zip(masks, names):
        f = cand_feats(m, n, lab, grad)
        if f is None: continue
        rows.append(f + [0, 0] + [1.0 * (n == c) for c in CHN] + [float(refined)]); sm.append(ds(m)); pk.append(pack(m)); nm.append(n)
    return rows, sm, pk, nm

def consensus(X, small):
    if len(small) == 0: return X
    sm = np.stack(small).reshape(len(small), -1).astype(np.float32); inter = sm @ sm.T; ar = sm.sum(1)
    iou = inter / (ar[:, None] + ar[None, :] - inter + 1e-9); np.fill_diagonal(iou, 0)
    X[:, 13] = (iou > 0.7).sum(1); X[:, 14] = (iou > 0.5).sum(1); return X

def build_pool_stage1(img):
    im_r, lab, grad = img_ctx(img); ms = candidate_masks(img)
    rows, sm, pk, nm = rows_for([m for _, m in ms], [n for n, _ in ms], lab, grad, 0)
    X = consensus(np.array(rows, float).reshape(-1, len(FEAT2)), sm)
    return dict(X=np.nan_to_num(X), small=sm, packed=pk, names=nm, shape=img.shape[:2])

def grabcut_refine(im_r, m):
    m8 = m.astype(np.uint8); H, W = m.shape
    for k_in in (21, 11):
        sure = cv2.erode(m8, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_in, k_in)))
        if sure.sum() >= 30: break
    else: return None
    near = cv2.dilate(m8, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (29, 29)))
    mask = np.full((H, W), cv2.GC_BGD, np.uint8); mask[near > 0] = cv2.GC_PR_BGD; mask[m] = cv2.GC_PR_FGD; mask[sure > 0] = cv2.GC_FGD
    try: cv2.grabCut(im_r, mask, None, np.zeros((1, 65)), np.zeros((1, 65)), 4, cv2.GC_INIT_WITH_MASK)
    except cv2.error: return None
    out = ndi.binary_fill_holes((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD)).astype(np.uint8)
    n, cc, st, _ = cv2.connectedComponentsWithStats(out, 8)
    if n < 2: return None
    return cc == (1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA])))

def add_refined(img, pool, scores1, topk=6):
    if len(pool["names"]) == 0: return pool
    im_r, lab, grad = img_ctx(img); H, W = img.shape[:2]; picks = []
    for i in np.argsort(-scores1):
        if all(iou_b(pool["small"][i], pool["small"][q]) < 0.85 for q in picks): picks.append(i)
        if len(picks) >= topk: break
    new = []
    for i in picks:
        g = grabcut_refine(im_r, ds(unpack(pool["packed"][i], pool["shape"]), R))
        if g is not None: new.append(cv2.resize(g.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR) > 0.5)
    rows, sm, pk, nm = rows_for(new, ["gc"] * len(new), lab, grad, 1)
    X = np.vstack([pool["X"], np.array(rows, float).reshape(-1, len(FEAT2))]) if rows else pool["X"].copy()
    small = pool["small"] + sm
    return dict(X=np.nan_to_num(consensus(X, small)), small=small, packed=pool["packed"] + pk, names=pool["names"] + nm, shape=pool["shape"])

def select_from_pool(d, sc, thr, nms=0.3):
    out = np.zeros(d["shape"], np.int32); kept, k = [], 0
    for j, i in enumerate(np.argsort(-sc)):
        if j > 0 and sc[i] < thr: break
        if all(iou_b(d["small"][i], d["small"][q]) < nms for q in kept):
            kept.append(i); m = unpack(d["packed"][i], d["shape"]); free = m & (out == 0)
            if free.sum() >= MIN_A * m.size: k += 1; out[free] = k
    return out

def segment_final(img, m1, m2, thr, topk=6):                           # the complete classical pipeline
    p1 = build_pool_stage1(img)
    if len(p1["names"]) == 0: return np.zeros(img.shape[:2], np.int32)
    p2 = add_refined(img, p1, m1.predict(p1["X"][:, :NF1]), topk)
    return select_from_pool(p2, m2.predict(p2["X"]), thr)

# ---------------- build the refined pool for all dev images ----------------
print("running GrabCut refinement on the top candidates of every dev image ...")
OOF4 = oof_scores(models["T4"][1])                                      # out-of-fold stage-1 scores (no label leakage)
t0 = time.time(); DATA2 = {}
for s in dev_stems:
    img, gt = DEV[s]; d = DATA[s]
    if len(d["names"]) == 0: DATA2[s] = dict(X=np.zeros((0, len(FEAT2))), y=np.zeros(0), small=[], packed=[], names=[], shape=d["shape"]); continue
    p1 = dict(X=np.hstack([d["X"], np.zeros((len(d["X"]), 1))]), small=d["small"], packed=d["packed"], names=d["names"], shape=d["shape"])
    p2 = add_refined(img, p1, OOF4[s])
    gts = [ds(gt == k) for k in range(1, int(gt.max()) + 1)]
    p2["y"] = np.array([max([iou_b(m, g) for g in gts], default=0.0) for m in p2["small"]]); DATA2[s] = p2
print(f"done in {time.time()-t0:.0f}s")

best2 = np.array([max([iou_b(m, g) for m in DATA2[s]["small"]], default=0.0) for s in dev_stems for g in DATA[s].get("gts", [])])
print(f"\nORACLE after adding refined proposals ({len(best2)} GT eggs):   before -> after")
for t in (0.5, 0.7, 0.8, 0.9): print(f"   IoU >= {t}: {np.mean(best >= t)*100:.0f}% -> {np.mean(best2 >= t)*100:.0f}%")
print(f"   mean best IoU: {best.mean():.3f} -> {best2.mean():.3f}")

X2 = np.vstack([DATA2[s]["X"] for s in dev_stems]); y2 = np.concatenate([DATA2[s]["y"] for s in dev_stems])
owner2 = np.concatenate([[i] * len(DATA2[s]["y"]) for i, s in enumerate(dev_stems)])
mk_rf = lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=0)
oof2 = {}
for tr, te in GroupKFold(5).split(dev_stems, groups=grp):
    mdl = mk_rf().fit(X2[np.isin(owner2, tr)], y2[np.isin(owner2, tr)])
    for i in te:
        X = DATA2[dev_stems[i]]["X"]; oof2[dev_stems[i]] = mdl.predict(X) if len(X) else np.zeros(0)
make_seg2 = lambda thr: (lambda img: select_from_pool(DATA2[ID2STEM[id(img)]], oof2[ID2STEM[id(img)]], thr))
grid = {t: evaluate(make_seg2(t), dev_stems) for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]}
bt = max(grid, key=lambda t: grid[t]["F1_at_0_5"]); seg_T5 = make_seg2(bt)
print("\nT5 threshold grid -> " + "  ".join(f"{t}:{grid[t]['F1_at_0_5']:.3f}" for t in grid) + f"   (best {bt})")
run_trial("T5", f"T4 + GrabCut-refined versions of the top-6 candidates added to the pool, RF re-ranks raw+refined (5-fold grouped CV), thr_add={bt}",
          seg_T5, "dev scores are out-of-fold")

# ---------------- save the complete pipeline (trained on all of dev) ----------------
m1 = mk_rf().fit(Xall, yall); m2 = mk_rf().fit(X2, y2)
joblib.dump(dict(m1=m1, m2=m2, thr=bt, topk=6, features=FEAT2), f"{PROJ}/task2/pipeline_models.joblib")
imp = pd.Series(m2.feature_importances_, FEAT2).sort_values(ascending=False).head(6); print("\nstage-2 top features:\n", imp.round(3).to_string())
_chk = [dev_stems[i] for i in (0, 1, 2)]
print("segment_final smoke test on 3 dev images (in-sample, just checking it runs):",
      [int(segment_final(DEV[s][0], m1, m2, bt).max()) for s in _chk], "eggs predicted; GT:", [int(DEV[s][1].max()) for s in _chk])
show_worst(seg_T5, "T5 worst dev images")

In [ ]:
import hashlib, json, datetime
# ---------------- 1) add keep/discard decisions + reasons to the experiment log ----------------
decisions = {
 "T0": ("DISCARD", "Assumed the egg is the bright region. Dataset has brown/dark eggs on bright walls, trays and cloth, so it picked the background."),
 "T1": ("DISCARD", "Whiteness cue is better than raw grey but still the same polarity assumption; same failure."),
 "T2": ("DISCARD", "Both polarities + 5 channels fixed the polarity problem and the count error (MAE 1.5 -> 0.46), but my hand-set score weights picked small corner regions over the big smooth egg."),
 "T3": ("DISCARD", "Learning the weights (linear) helped a lot (F1 0.49 -> 0.68) but a linear score cannot express interactions such as 'large AND smooth AND high contrast'."),
 "T4": ("SUPERSEDED", "Random forest ranker: F1 0.715. Oracle showed proposals themselves were the ceiling (only 45% of eggs had a candidate at IoU>=0.8)."),
 "T5": ("KEEP (FINAL)", "GrabCut-refined candidates added to the pool: oracle IoU>=0.8 rose 45% -> 53%, matched-egg IoU 0.80 -> 0.81, F1 unchanged. Frozen as the final method."),
}
for r in exp_log: r["decision"], r["reason"] = decisions[r["trial"]]
exp_log.sort(key=lambda r: r["trial"]); log_df = pd.DataFrame(exp_log); log_df.to_csv(LOG_PATH, index=False)
print(log_df[["trial", "F1_at_0_5", "mean_inst_IoU", "fg_IoU", "count_MAE", "decision"]].to_string(index=False))

# ---------------- 2) freeze: all hyper-parameters + hash of the trained models ----------------
mp = f"{PROJ}/task2/pipeline_models.joblib"
cfg = dict(
    frozen_at=datetime.datetime.now().isoformat(timespec="seconds"), final_trial="T5", models_sha256=hashlib.sha256(open(mp, "rb").read()).hexdigest(),
    split=dict(dev_images=len(dev_stems), hash_threshold=8, group_split_seed=43),
    proposals=dict(channels=["L", "a", "b", "S", "whiteness(L-1.5*chroma)"], threshold="Otsu, both polarities", gaussian_blur=7,
                   morphology="open+close 9x9 ellipse", min_area_frac=MIN_A, fill_holes=True),
    features=FEAT2, feature_resolution=R,
    ranker=dict(type="RandomForestRegressor", n_estimators=300, min_samples_leaf=3, random_state=0, target="IoU with GT egg"),
    refinement=dict(method="GrabCut", iterations=4, sure_fg_erosion_px=[21, 11], probable_bg_dilation_px=29, topk=6, work_resolution=R),
    selection=dict(first_always_kept=True, others_if_score_ge=float(bt), nms_iou=0.3),
    dev_scores_out_of_fold=exp_log[-1] if exp_log[-1]["trial"] == "T5" else [r for r in exp_log if r["trial"] == "T5"][0])
json.dump(cfg, open(f"{PROJ}/task2/final_config.json", "w"), indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
print("\nFROZEN. models sha256:", cfg["models_sha256"][:16], "...  threshold:", bt)

# ---------------- 3) back up ----------------
for d in ["task1", "splits", "task2"]: shutil.copytree(f"{PROJ}/{d}", f"/content/drive/MyDrive/Nosh_Egg_Project_Backup/{d}", dirs_exist_ok=True)
print("backed up task2 to Drive")

In [ ]:
# ============================ TASK 3: one-shot evaluation on the held-out eval split ============================
N_EVAL_EXPECTED = 32
import hashlib, json
cfg = json.load(open(f"{PROJ}/task2/final_config.json")); mp = f"{PROJ}/task2/pipeline_models.joblib"
assert hashlib.sha256(open(mp, "rb").read()).hexdigest() == cfg["models_sha256"], "models changed after the freeze - stop"
bundle = joblib.load(mp); m1, m2, thr, topk = bundle["m1"], bundle["m2"], bundle["thr"], bundle["topk"]
eval_stems = t1[t1.split == "eval"].stem.tolist(); assert len(eval_stems) == N_EVAL_EXPECTED, len(eval_stems)
os.makedirs(f"{PROJ}/task3/pred", exist_ok=True)

EVAL = {}; t0 = time.time()
for s in eval_stems:
    img = cv2.cvtColor(cv2.imread(_path(s)), cv2.COLOR_BGR2RGB); gt = np.load(f"{PROJ}/task1/labels/{s}.npy"); pf = f"{PROJ}/task3/pred/{s}.npy"
    if os.path.exists(pf): pr = np.load(pf)                        # predictions are computed ONCE and then reused
    else: pr = segment_final(img, m1, m2, thr, topk); np.save(pf, pr)
    EVAL[s] = (img, gt, pr)
print(f"predictions for {len(EVAL)} eval images ready ({time.time()-t0:.0f}s)")

# ---------------- analysis of one image: matches + failure taxonomy ----------------
def analyse_image(gt, pr):
    g, p = int(gt.max()), int(pr.max())
    ga = np.array([(gt == k).sum() for k in range(1, g + 1)], float); pa = np.array([(pr == k).sum() for k in range(1, p + 1)], float)
    inter = np.zeros((g, p)); iou = np.zeros((g, p))
    if g and p:
        pair = np.bincount(gt.ravel() * (p + 1) + pr.ravel(), minlength=(g + 1) * (p + 1)).reshape(g + 1, p + 1)
        inter = pair[1:, 1:].astype(float); iou = inter / (ga[:, None] + pa[None, :] - inter + 1e-9)
    match = {}
    if g and p:
        r, c = linear_sum_assignment(-iou); match = {int(i): int(j) for i, j in zip(r, c) if iou[i, j] >= 0.5}
    cats, ious = [], []
    for i in range(g):
        if i in match:
            v = iou[i, match[i]]; ious.append(v); cats.append("good (IoU>=0.75)" if v >= 0.75 else "boundary error (0.5<=IoU<0.75)"); continue
        ious.append(iou[i].max() if p else 0.0)
        if p == 0: cats.append("miss (nothing predicted)"); continue
        j = int(np.argmax(inter[i])); cover = inter[i, j] / ga[i]; prec = inter[i, j] / pa[j]
        if (inter[:, j] / ga > 0.3).sum() >= 2: cats.append("merge (one region covers 2+ eggs)")
        elif (inter[i] / ga[i] > 0.2).sum() >= 2: cats.append("split (egg broken into 2+ regions)")
        elif cover < 0.1: cats.append("miss (wrong object chosen)")
        elif cover >= 0.5 and prec < 0.5: cats.append("leak (region far larger than egg)")
        elif prec >= 0.5 and cover < 0.5: cats.append("partial (only part of the egg)")
        else: cats.append("mislocalised")
    matched_preds = set(match.values()); fp = 0
    for j in range(p):
        if j in matched_preds: continue
        if g == 0 or inter[:, j].sum() / pa[j] < 0.3: fp += 1
    return dict(g=g, p=p, ious=ious, cats=cats, fp=fp, ga=ga)

rows, egg_rows = [], []
for s in eval_stems:
    img, gt, pr = EVAL[s]; a = analyse_image(gt, pr); A, B = gt > 0, pr > 0
    rows.append(dict(stem=s, n_gt=a["g"], n_pred=a["p"], fg_iou=(A & B).sum() / ((A | B).sum() + 1e-9) if A.any() else np.nan,
                     fg_dice=2 * (A & B).sum() / (A.sum() + B.sum() + 1e-9) if A.any() else np.nan, false_pos=a["fp"],
                     tp50=sum(v >= .5 for v, c in zip(a["ious"], a["cats"]) if c.startswith(("good", "boundary"))),
                     tp75=sum(v >= .75 for v in a["ious"])))
    for k, (v, c) in enumerate(zip(a["ious"], a["cats"])): egg_rows.append(dict(stem=s, egg=k + 1, iou=v, category=c, area_frac=a["ga"][k] / gt.size))
per = pd.DataFrame(rows); eggs = pd.DataFrame(egg_rows)
per.to_csv(f"{PROJ}/task3/eval_per_image.csv", index=False); eggs.to_csv(f"{PROJ}/task3/eval_per_egg.csv", index=False)

def summarise(per):
    G, P = per.n_gt.sum(), per.n_pred.sum(); tp, tp75 = per.tp50.sum(), per.tp75.sum()
    pr_, rc_ = tp / max(P, 1), tp / max(G, 1); pr75, rc75 = tp75 / max(P, 1), tp75 / max(G, 1)
    has = per.n_gt > 0
    return dict(precision=pr_, recall=rc_, F1_at_0_5=2 * pr_ * rc_ / max(pr_ + rc_, 1e-9), F1_at_0_75=2 * pr75 * rc75 / max(pr75 + rc75, 1e-9),
                fg_IoU=per.fg_iou[has].mean(), fg_Dice=per.fg_dice[has].mean(), count_MAE=(per.n_gt - per.n_pred).abs().mean(),
                exact_count_acc=(per.n_gt == per.n_pred).mean(), false_positive_regions=int(per.false_pos.sum()))
res = summarise(per); rng_ = np.random.default_rng(0); boots = []
for _ in range(1000): boots.append(summarise(per.iloc[rng_.integers(0, len(per), len(per))].reset_index(drop=True)))
ci = {k: (np.percentile([b[k] for b in boots], 2.5), np.percentile([b[k] for b in boots], 97.5)) for k in ("F1_at_0_5", "fg_IoU", "fg_Dice")}
dev_row = [r for r in exp_log if r["trial"] == "T5"][0]
ex = eggs[eggs.category.str.startswith(("good", "boundary"))]
print(f"\n===== EVAL RESULTS ({len(per)} images, {int(per.n_gt.sum())} GT eggs, {int(per.n_pred.sum())} predicted) =====")
tbl = pd.DataFrame({"eval (final, frozen)": res, "dev (5-fold out-of-fold)": {"precision": dev_row["precision"], "recall": dev_row["recall"], "F1_at_0_5": dev_row["F1_at_0_5"],
                    "fg_IoU": dev_row["fg_IoU"], "fg_Dice": dev_row["fg_Dice"], "count_MAE": dev_row["count_MAE"]}})
print(tbl.round(3).to_string())
print(f"\nmean IoU of correctly matched eggs (IoU>=0.5): {ex.iou.mean():.3f}")
for k, (lo, hi) in ci.items(): print(f"95% bootstrap CI  {k}: {res[k]:.3f}  [{lo:.3f}, {hi:.3f}]")
print("\nFAILURE TAXONOMY (per GT egg) + false positives:"); tax = eggs.category.value_counts(); print(tax.to_string()); print(f"false-positive regions: {int(per.false_pos.sum())}")
json.dump(dict(eval=res, ci={k: list(v) for k, v in ci.items()}, taxonomy=tax.to_dict(), n_images=len(per), n_gt=int(per.n_gt.sum())),
          open(f"{PROJ}/task3/eval_metrics.json", "w"), indent=2, default=float)

# ---------------- plots ----------------
def contour_vis(img, gt, pr):
    vis = img.copy()
    for m_, col in [(gt > 0, (0, 255, 0)), (pr > 0, (255, 0, 0))]:
        cs, _ = cv2.findContours(m_.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(vis, cs, -1, col, 3)
    return vis
fig, axs = plt.subplots(1, 4, figsize=(24, 5))
axs[0].hist(per.fg_iou.dropna(), bins=np.linspace(0, 1, 11), color="#4C72B0"); axs[0].set_title("per-image foreground IoU"); axs[0].set_xlabel("IoU"); axs[0].set_ylabel("images")
axs[1].scatter(eggs.area_frac, eggs.iou, c="#C44E52"); axs[1].axhline(.5, ls="--", c="gray"); axs[1].set_xlabel("egg area / image area"); axs[1].set_ylabel("IoU with best prediction"); axs[1].set_title("does egg size matter?")
tax.sort_values().plot.barh(ax=axs[2], color="#55A868"); axs[2].set_title("failure taxonomy (per GT egg)")
axs[3].scatter(per.n_gt + rng_.normal(0, .05, len(per)), per.n_pred + rng_.normal(0, .05, len(per)), c="#8172B2"); mx = max(per.n_gt.max(), per.n_pred.max()) + 1
axs[3].plot([0, mx], [0, mx], "k--"); axs[3].set_xlabel("GT egg count"); axs[3].set_ylabel("predicted count"); axs[3].set_title("count agreement")
plt.tight_layout(); plt.savefig(f"{PROJ}/task3/analysis_plots.png", dpi=130); plt.show()

def grid_fig(stems, title, path):
    n = len(stems); cols = 4; rows_ = int(np.ceil(n / cols)); fig, axs = plt.subplots(rows_, cols, figsize=(22, 5.6 * rows_)); axs = np.atleast_1d(axs).ravel()
    for a in axs: a.axis("off")
    for a, s in zip(axs, stems):
        img, gt, pr = EVAL[s]; r_ = per.set_index("stem").loc[s]; a.imshow(contour_vis(img, gt, pr))
        cat = ", ".join(sorted(set(eggs[eggs.stem == s].category.str.split(" ").str[0]))) or "no eggs"
        a.set_title(f"{s[:24]}  GT={int(r_.n_gt)} pred={int(r_.n_pred)} fgIoU={r_.fg_iou:.2f}\n{cat}", fontsize=10)
    plt.suptitle(title + "   green = ground truth (SAM + review), red = classical prediction", fontsize=15); plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()
srt = per.assign(k=per.fg_iou.fillna(1.0)).sort_values("k")
grid_fig(srt.stem.head(8).tolist(), "WORST 8 eval images", f"{PROJ}/task3/worst_cases.png")
grid_fig(srt.stem.tail(4).tolist(), "BEST 4 eval images", f"{PROJ}/task3/best_cases.png")
for d in ["task3"]: shutil.copytree(f"{PROJ}/{d}", f"/content/drive/MyDrive/Nosh_Egg_Project_Backup/{d}", dirs_exist_ok=True)

In [ ]:
# ---------------- Task 3 extras: does egg size matter? + Task 2 experiment-log table for the report ----------------
eggs["size_bin"] = pd.qcut(eggs.area_frac, 3, labels=["small", "medium", "large"])
eggs["ok"] = eggs.category.str.startswith(("good", "boundary"))
sz = eggs.groupby("size_bin", observed=True).agg(eggs=("ok", "size"), found_IoU_ge_0_5=("ok", "mean"), median_IoU=("iou", "median"), area_lo=("area_frac", "min"), area_hi=("area_frac", "max"))
print("by egg size (share of image area):\n", sz.round(2).to_string())
print("\nfailure share of all", len(eggs), "GT eggs:\n", (eggs.category.value_counts(normalize=True) * 100).round(0).astype(int).astype(str).add(" %").to_string())

params = {
 "T0": ("Baseline: grey, global Otsu", "simplest classical pipeline; establishes a floor", "Gaussian 5x5; Otsu; open+close 5x5 ellipse; fill holes; min area 500 px"),
 "T1": ("Colour cue: LAB whiteness = L - 1.5*chroma", "eggs look white/low-saturation; wanted to ignore coloured background", "as T0, whiteness map instead of grey"),
 "T2": ("Polarity-agnostic: 5 channels x 2 polarities, rank candidates by hand-set score", "T0/T1 showed eggs are often darker than background", "channels L,a,b,S,whiteness; Gaussian 7; Otsu hi/lo; open+close 9x9; min area 2%; score=0.4*solidity+0.4*edge/(edge+15)+0.2/(1+colour std/10); NMS IoU 0.3; thr_add 0.65 (grid 0.55-1.01)"),
 "T3": ("Learn the score weights (Ridge)", "hand weights picked corner regions; let dev data choose", f"{NF1} features; Ridge alpha 10 on standardised features; target = IoU with GT egg; 5-fold GroupKFold on pHash groups; thr_add 0.5"),
 "T4": ("Non-linear ranker (Random Forest)", "linear score cannot express 'large AND smooth AND high contrast'", "RF 300 trees, min_samples_leaf 3, seed 0; same features/CV; thr_add 0.5"),
 "T5": ("Add GrabCut-refined candidates", "oracle showed proposals (not ranking) limited quality: only 45% of eggs had a candidate at IoU>=0.8", "GrabCut 4 iterations at 320 px; sure-FG = candidate eroded 21 px (11 fallback); probable-BG = dilated 29 px; top-6 candidates; second RF over raw+refined; thr_add 0.5"),
}
L_ = ["| Trial | What changed | Why tried | Parameters | Dev precision | Dev recall | Dev F1@0.5 | Dev matched IoU | Dev fg IoU | Count MAE | Decision and reason |", "|---|---|---|---|---|---|---|---|---|---|---|"]
for r in sorted(exp_log, key=lambda r: r["trial"]):
    w_, y_, p_ = params[r["trial"]]
    L_.append(f"| {r['trial']} | {w_} | {y_} | {p_} | {r['precision']:.3f} | {r['recall']:.3f} | {r['F1_at_0_5']:.3f} | {r['mean_inst_IoU']:.3f} | {r['fg_IoU']:.3f} | {r['count_MAE']:.2f} | **{r['decision']}**: {r['reason']} |")
open(f"{PROJ}/task2/experiment_log.md", "w").write("\n".join(L_) + "\n"); print("\nsaved task2/experiment_log.md (paste into the report)")
shutil.copytree(f"{PROJ}/task2", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task2", dirs_exist_ok=True)

In [ ]:
# ============================ TASK 4: get the video and look at it ============================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp"], check=True)
os.makedirs(f"{PROJ}/task4", exist_ok=True); VID = f"{PROJ}/task4/source.mp4"; URL = "https://www.youtube.com/watch?v=Y4JyV94y2_A"
if not os.path.exists(VID):
    r = subprocess.run(["yt-dlp", "-f", "bv*[height<=720][ext=mp4]+ba[ext=m4a]/b[height<=720][ext=mp4]/b[height<=720]", "--merge-output-format", "mp4", "-o", VID, URL], capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])
assert os.path.exists(VID), "download failed - upload the video file to task4/source.mp4 instead (see message)"
cap = cv2.VideoCapture(VID); fps = cap.get(cv2.CAP_PROP_FPS); nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); vw, vh = int(cap.get(3)), int(cap.get(4))
print(f"video: {vw}x{vh}  {fps:.1f} fps  {nfr} frames  {nfr/fps:.1f} s")

def read_frame(i, long_side=640):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, f = cap.read(); f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB); s = long_side / max(f.shape[:2])
    return cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
idx = np.linspace(0, nfr - 1, 8).astype(int); frames = [read_frame(i) for i in idx]
fig, axs = plt.subplots(2, 4, figsize=(22, 9))
for a, f, i in zip(axs.ravel(), frames, idx): a.imshow(f); a.set_title(f"frame {i}  ({i/fps:.1f}s)"); a.axis("off")
plt.suptitle("8 evenly spaced frames of the task video"); plt.tight_layout(); plt.show()

# how does the frozen Task 2 pipeline do on video frames out of the box?
fig, axs = plt.subplots(2, 4, figsize=(22, 9)); t0 = time.time()
for k, (f, i) in enumerate(list(zip(frames, idx))[::2]):
    lab_ = segment_final(f, m1, m2, thr, topk)
    axs[0, k].imshow(f); axs[0, k].axis("off"); axs[1, k].imshow(overlay(f, lab_)); axs[1, k].set_title(f"frozen pipeline: {int(lab_.max())} regions"); axs[1, k].axis("off")
print(f"frozen pipeline: {(time.time()-t0)/4:.1f} s per frame"); plt.tight_layout(); plt.show()

In [ ]:
# ============================ TASK 4 / step 1: video-specific classical egg segmenter ============================
LONG = 800
def prep(f):
    s = LONG / max(f.shape[:2]); return cv2.resize(f, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

V = dict(L_min=175, chroma_max=28, open_frac=0.022, area_min=0.003, area_max=0.15, solidity_min=0.88, ellipse_iou_min=0.80, aspect_max=2.0, split_peak=0.75)

def split_touching(comp, img_bgr):
    """distance-transform watershed: split one blob into several eggs if its distance map has several clear peaks"""
    m8 = comp.astype(np.uint8); dist = cv2.distanceTransform(m8, cv2.DIST_L2, 5)
    for frac in (V["split_peak"], 0.9):                                 # first a moderate, then a strict peak threshold (for heavier overlap)
        n, pk = cv2.connectedComponents((dist > frac * dist.max()).astype(np.uint8))
        if n > 2: break
    if n <= 2: return [comp]
    markers = np.zeros(comp.shape, np.int32); markers[~comp] = 1
    for k in range(1, n): markers[pk == k] = k + 1
    cv2.watershed(img_bgr, markers)
    return [(markers == k + 1) & comp for k in range(1, n) if ((markers == k + 1) & comp).sum() > 0]

def video_seg(f):
    """f: RGB frame -> label map (0 background, 1..N eggs). Pure classical: whiteness threshold, morphology, watershed, shape filters."""
    H, W = f.shape[:2]; lab = cv2.cvtColor(cv2.GaussianBlur(f, (5, 5), 0), cv2.COLOR_RGB2LAB).astype(np.float32)
    chroma = np.hypot(lab[..., 1] - 128, lab[..., 2] - 128)
    m = ((lab[..., 0] > V["L_min"]) & (chroma < V["chroma_max"])).astype(np.uint8)
    k = int(V["open_frac"] * W) | 1; m = cv2.morphologyEx(m, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    m = ndi.binary_fill_holes(m).astype(np.uint8); n, cc, st, _ = cv2.connectedComponentsWithStats(m, 8)
    bgr = cv2.cvtColor(f, cv2.COLOR_RGB2BGR); out = np.zeros((H, W), np.int32); idx = 0
    for i in range(1, n):
        for part in split_touching(cc == i, bgr):
            a = part.sum() / part.size
            if not (V["area_min"] <= a <= V["area_max"]): continue
            cs, _ = cv2.findContours(part.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); c = max(cs, key=cv2.contourArea)
            if len(c) < 5: continue
            sol = cv2.contourArea(c) / (cv2.contourArea(cv2.convexHull(c)) + 1e-9)
            (cx, cy), (ma, mi), ang = cv2.fitEllipse(c); em = np.zeros((H, W), np.uint8); cv2.ellipse(em, ((cx, cy), (ma, mi), ang), 1, -1); em = em.astype(bool)
            eiou = (em & part).sum() / ((em | part).sum() + 1e-9)
            if sol >= V["solidity_min"] and eiou >= V["ellipse_iou_min"] and max(ma, mi) / max(min(ma, mi), 1) <= V["aspect_max"]:
                idx += 1; out[part] = idx
    return out

# ---------------- preview on frames from the egg scene and from the title/text slides ----------------
pick = list(np.linspace(int(12 * fps), int(40 * fps), 10).astype(int)) + [int(1 * fps), int(45 * fps)]
fig, axs = plt.subplots(3, 4, figsize=(22, 11)); t0 = time.time(); counts = []
for a, i in zip(axs.ravel(), pick):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap.read(); fr = prep(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); lb = video_seg(fr); counts.append(int(lb.max()))
    a.imshow(overlay(fr, lb)); a.set_title(f"frame {i} ({i/fps:.1f}s): {int(lb.max())} eggs"); a.axis("off")
print(f"{(time.time()-t0)/len(pick)*1000:.0f} ms per frame;  egg counts on the 12 preview frames: {counts}")
plt.suptitle("video segmenter preview (last 2 panels = frames WITHOUT eggs: should show 0)"); plt.tight_layout(); plt.show()

In [ ]:
# ============================ TASK 4 / step 2: tracker + labelled output video ============================
class Track:
    def __init__(s, mask, c, area): s.mask, s.c, s.area, s.v = mask, np.array(c, float), float(area), np.zeros(2); s.hits, s.misses, s.id, s.seen = 1, 0, None, True

def mask_shift(m, v):
    M = np.float32([[1, 0, v[0]], [0, 1, v[1]]]); return cv2.warpAffine(m.astype(np.uint8), M, (m.shape[1], m.shape[0]), flags=cv2.INTER_NEAREST) > 0

def detections(lb):
    ds_ = []
    for k in range(1, int(lb.max()) + 1):
        m = lb == k; ys, xs = np.nonzero(m)
        if len(xs): ds_.append(dict(mask=m, c=np.array([xs.mean(), ys.mean()]), area=float(m.sum())))
    return ds_

def associate(tracks, dets, big=1e6):
    """Hungarian assignment; cost = mix of (1 - IoU with the motion-shifted track mask) and centroid distance, gated by egg size and speed"""
    if not tracks or not dets: return []
    C = np.full((len(tracks), len(dets)), big)
    for i, t in enumerate(tracks):
        pm = mask_shift(t.mask, t.v); pc = t.c + t.v; gate = 3.0 * np.sqrt(t.area) + 2.0 * np.linalg.norm(t.v)
        for j, d in enumerate(dets):
            iou = (pm & d["mask"]).sum() / ((pm | d["mask"]).sum() + 1e-9); dist = np.linalg.norm(pc - d["c"])
            if iou > 0 or dist < gate: C[i, j] = 0.5 * (1 - iou) + 0.5 * min(dist / gate, 1.0)
    r, c = linear_sum_assignment(C); return [(i, j) for i, j in zip(r, c) if C[i, j] < big / 2]

MIN_SEEN_S = 1.0
PALETTE = [(255, 80, 80), (60, 200, 255), (120, 255, 120), (255, 200, 60), (200, 120, 255), (255, 120, 200)]
def run_tracker(frame_iter, fps, writer=None, keep=None, max_age_s=1.5, confirm_hits=5, cut_thresh=40.0):
    tracks, next_id, rows, prev_small = [], 1, [], None; max_age = int(max_age_s * fps); trails = {}
    for fi, fr in enumerate(frame_iter):
        small = cv2.resize(fr, (64, 36)).astype(np.float32)
        if prev_small is not None and np.abs(small - prev_small).mean() > cut_thresh: tracks, trails = [], {}      # hard scene cut -> forget tracks
        prev_small = small
        dets = detections(video_seg(fr)); pairs = associate(tracks, dets); mt = {i: j for i, j in pairs}; md = {j for _, j in pairs}
        for i, t in enumerate(tracks):
            if i in mt:
                d = dets[mt[i]]; t.v = 0.5 * t.v + 0.5 * (d["c"] - t.c); t.c, t.mask, t.area = d["c"], d["mask"], 0.7 * t.area + 0.3 * d["area"]; t.hits += 1; t.misses = 0; t.seen = True
                if t.id is None and t.hits >= confirm_hits: t.id = next_id; next_id += 1
            else:
                t.misses += 1; t.seen = False; t.c = t.c + t.v; t.mask = mask_shift(t.mask, t.v); t.v *= 0.8                 # coast through occlusion
        tracks = [t for t in tracks if (t.id is not None and t.misses <= max_age) or (t.id is None and t.misses == 0)]
        tracks += [Track(d["mask"], d["c"], d["area"]) for j, d in enumerate(dets) if j not in md]
        vis = fr.copy()
        n_vis = 0
        for t in tracks:
            if t.id is None or (keep is not None and t.id not in keep): continue
            did = t.id if keep is None else keep[t.id]; n_vis += int(t.seen)
            rows.append(dict(frame=fi, time_s=fi / fps, id=did, cx=t.c[0], cy=t.c[1], area=t.area, detected=t.seen))
            col = PALETTE[(did - 1) % len(PALETTE)]; trails.setdefault(t.id, []).append((int(t.c[0]), int(t.c[1])))
            if t.seen:
                ov = vis.copy(); ov[t.mask] = col; vis = cv2.addWeighted(ov, 0.45, vis, 0.55, 0)
                cs, _ = cv2.findContours(t.mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(vis, cs, -1, col, 2); label = f"egg #{did}"
            elif t.misses <= int(0.5 * fps):
                ys, xs = np.nonzero(t.mask)
                if len(xs): cv2.rectangle(vis, (xs.min(), ys.min()), (xs.max(), ys.max()), col, 1); label = f"egg #{did} (occluded)"
                else: continue
            else: continue
            x0, y0 = int(t.c[0]) - 30, int(t.c[1]) - int(np.sqrt(t.area)) - 8
            cv2.putText(vis, label, (max(x0, 2), max(y0, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 4); cv2.putText(vis, label, (max(x0, 2), max(y0, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
            tr = trails[t.id][-40:]
            for a_, b_ in zip(tr[:-1], tr[1:]): cv2.line(vis, a_, b_, col, 2)
        cv2.putText(vis, f"t={fi/fps:5.1f}s  eggs tracked: {n_vis}", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4); cv2.putText(vis, f"t={fi/fps:5.1f}s  eggs tracked: {n_vis}", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        if writer is not None: writer.write(cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return pd.DataFrame(rows)

def frames_from_cap(cap_):
    cap_.set(cv2.CAP_PROP_POS_FRAMES, 0)
    while True:
        ok, f = cap_.read()
        if not ok: break
        yield prep(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))

first = prep(read_frame(0, LONG)); Hh, Ww = first.shape[:2]
raw_out = f"{PROJ}/task4/tracked_raw.mp4"; final_out = f"{PROJ}/task4/egg_tracking_output.mp4"
t0 = time.time(); pass1 = run_tracker(frames_from_cap(cap), fps)                       # pass 1: find all confirmed tracks, no drawing
seen_s = pass1.groupby("id").detected.sum() / fps; first_t = pass1.groupby("id").time_s.min()
good = [i for i in seen_s.index if seen_s[i] >= MIN_SEEN_S]; good.sort(key=lambda i: first_t[i]); keep = {old: new + 1 for new, old in enumerate(good)}
print(f"pass 1: {len(seen_s)} confirmed tracks, {len(good)} kept (seen >= {MIN_SEEN_S}s); dropped short ghost tracks: {len(seen_s) - len(good)}   [{time.time()-t0:.0f}s]")
wr = cv2.VideoWriter(raw_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (Ww, Hh))
trk = run_tracker(frames_from_cap(cap), fps, wr, keep=keep); wr.release()                # pass 2: identical tracking, now drawing only the kept tracks
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_out, "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", "-movflags", "+faststart", final_out], check=True)
trk.to_csv(f"{PROJ}/task4/tracks.csv", index=False)
summ = trk.groupby("id").agg(first_s=("time_s", "min"), last_s=("time_s", "max"), frames=("frame", "count"), detected=("detected", "sum"), mean_area=("area", "mean")).round(2)
print("\nfinal tracks:\n", summ.to_string()); print("\nvideo:", final_out, f"  [{time.time()-t0:.0f}s total]")
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5))
for tid, g in trk.groupby("id"):
    axs[0].plot(g.time_s, g.cx, label=f"egg #{tid}"); axs[1].plot(g.time_s, g.cy, label=f"egg #{tid}")
axs[0].set_title("x position of each tracked egg"); axs[1].set_title("y position of each tracked egg")
for a in axs: a.set_xlabel("time (s)"); a.legend()
plt.tight_layout(); plt.savefig(f"{PROJ}/task4/trajectories.png", dpi=120); plt.show()
os.makedirs("/content/drive/MyDrive/Nosh_Egg_Project_Backup/task4", exist_ok=True)
for fn in ["egg_tracking_output.mp4", "tracks.csv", "trajectories.png"]: shutil.copy(f"{PROJ}/task4/{fn}", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task4/" + fn)
print("copied to Drive: Nosh_Egg_Project_Backup/task4/")
# contact sheet of the output so you can check it without opening the video
cap2 = cv2.VideoCapture(final_out); n2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT)); sel = np.linspace(int(10 * fps), int(42 * fps), 8).astype(int); fig, axs = plt.subplots(2, 4, figsize=(22, 8))
for a, i in zip(axs.ravel(), sel): cap2.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap2.read(); a.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); a.set_title(f"output frame {i} ({i/fps:.1f}s)"); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ============================ TASK 4 / step 2 (v2): tracker + labelled output video ============================
V["area_min"] = 0.008          # was 0.003: removes the small ghost regions on the shirt/hand (eggs are ~0.017-0.022 of the frame)
class Track:
    def __init__(s, mask, c, area): s.mask, s.c, s.area, s.v = mask, np.array(c, float), float(area), np.zeros(2); s.hits, s.misses, s.id, s.seen = 1, 0, None, True

def mask_shift(m, v):
    M = np.float32([[1, 0, v[0]], [0, 1, v[1]]]); return cv2.warpAffine(m.astype(np.uint8), M, (m.shape[1], m.shape[0]), flags=cv2.INTER_NEAREST) > 0

def detections(lb):
    ds_ = []
    for k in range(1, int(lb.max()) + 1):
        m = lb == k; ys, xs = np.nonzero(m)
        if len(xs): ds_.append(dict(mask=m, c=np.array([xs.mean(), ys.mean()]), area=float(m.sum())))
    return ds_

def associate(tracks, dets, big=1e6):
    """Hungarian assignment. cost = 0.4*(1-IoU with motion-shifted mask) + 0.4*centroid distance + 0.2*(1-area ratio);
    the distance gate grows the longer a track has been lost, so an egg that reappears after being hidden by the hand is re-linked."""
    if not tracks or not dets: return []
    C = np.full((len(tracks), len(dets)), big)
    for i, t in enumerate(tracks):
        pm = mask_shift(t.mask, t.v); pc = t.c + t.v; gate = (3.0 + 0.25 * min(t.misses, 20)) * np.sqrt(t.area) + 2.0 * np.linalg.norm(t.v)
        for j, d in enumerate(dets):
            iou = (pm & d["mask"]).sum() / ((pm | d["mask"]).sum() + 1e-9); dist = np.linalg.norm(pc - d["c"]); ar = min(t.area, d["area"]) / max(t.area, d["area"])
            if iou > 0 or dist < gate: C[i, j] = 0.4 * (1 - iou) + 0.4 * min(dist / gate, 1.0) + 0.2 * (1 - ar)
    r, c = linear_sum_assignment(C); return [(i, j) for i, j in zip(r, c) if C[i, j] < big / 2]

MIN_SEEN_S = 1.0
PALETTE = [(255, 80, 80), (60, 200, 255), (120, 255, 120), (255, 200, 60), (200, 120, 255), (255, 120, 200)]
def run_tracker(frame_iter, fps, writer=None, keep=None, max_age_s=8.0, confirm_hits=5, cut_thresh=40.0, scene_diff=0.5):
    tracks, next_id, rows, prev_small, ref_hist = [], 1, [], None, None; max_age = int(max_age_s * fps); trails = {}
    for fi, fr in enumerate(frame_iter):
        small = cv2.resize(fr, (64, 36)).astype(np.float32)
        if prev_small is not None and np.abs(small - prev_small).mean() > cut_thresh: tracks, trails = [], {}      # hard scene cut -> forget tracks
        prev_small = small
        dets = detections(video_seg(fr))
        hist = np.bincount((small.mean(2) / 8).astype(int).ravel().clip(0, 31), minlength=32) / small[..., 0].size            # 32-bin brightness histogram of the frame
        if dets: ref_hist = hist if ref_hist is None else 0.95 * ref_hist + 0.05 * hist                                   # what the egg scene normally looks like
        elif ref_hist is not None and 0.5 * np.abs(hist - ref_hist).sum() > scene_diff: tracks, trails, ref_hist = [], {}, None  # scene faded to a title/text slide -> stop tracking
        pairs = associate(tracks, dets); mt = {i: j for i, j in pairs}; md = {j for _, j in pairs}
        for i, t in enumerate(tracks):
            if i in mt:
                d = dets[mt[i]]; t.v = 0.5 * t.v + 0.5 * (d["c"] - t.c); vmax = 0.5 * np.sqrt(t.area); t.v = t.v * min(1.0, vmax / (np.linalg.norm(t.v) + 1e-9)); t.c, t.mask, t.area = d["c"], d["mask"], 0.7 * t.area + 0.3 * d["area"]; t.hits += 1; t.misses = 0; t.seen = True
                if t.id is None and t.hits >= confirm_hits: t.id = next_id; next_id += 1
            else:
                t.misses += 1; t.seen = False; t.c = t.c + t.v; t.mask = mask_shift(t.mask, t.v); t.v *= 0.5                 # coast through occlusion (velocity capped and damped so it cannot fly off)
        tracks = [t for t in tracks if (t.id is not None and t.misses <= max_age) or (t.id is None and t.misses == 0)]
        tracks += [Track(d["mask"], d["c"], d["area"]) for j, d in enumerate(dets) if j not in md]
        vis = fr.copy()
        n_vis = 0
        for t in tracks:
            if t.id is None or (keep is not None and t.id not in keep): continue
            did = t.id if keep is None else keep[t.id]; n_vis += int(t.seen)
            rows.append(dict(frame=fi, time_s=fi / fps, id=did, cx=t.c[0], cy=t.c[1], area=t.area, detected=t.seen))
            col = PALETTE[(did - 1) % len(PALETTE)]; trails.setdefault(t.id, []).append((int(t.c[0]), int(t.c[1])))
            if t.seen:
                ov = vis.copy(); ov[t.mask] = col; vis = cv2.addWeighted(ov, 0.45, vis, 0.55, 0)
                cs, _ = cv2.findContours(t.mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(vis, cs, -1, col, 2); label = f"egg #{did}"
            elif t.misses <= int(0.5 * fps):
                ys, xs = np.nonzero(t.mask)
                if len(xs): cv2.rectangle(vis, (xs.min(), ys.min()), (xs.max(), ys.max()), col, 1); label = f"egg #{did} (occluded)"
                else: continue
            else: continue
            x0, y0 = int(t.c[0]) - 30, int(t.c[1]) - int(np.sqrt(t.area)) - 8
            cv2.putText(vis, label, (max(x0, 2), max(y0, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 4); cv2.putText(vis, label, (max(x0, 2), max(y0, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
            tr = trails[t.id][-40:]
            for a_, b_ in zip(tr[:-1], tr[1:]): cv2.line(vis, a_, b_, col, 2)
        cv2.putText(vis, f"t={fi/fps:5.1f}s  eggs tracked: {n_vis}", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4); cv2.putText(vis, f"t={fi/fps:5.1f}s  eggs tracked: {n_vis}", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        if writer is not None: writer.write(cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return pd.DataFrame(rows)

def frames_from_cap(cap_):
    cap_.set(cv2.CAP_PROP_POS_FRAMES, 0)
    while True:
        ok, f = cap_.read()
        if not ok: break
        yield prep(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))

first = prep(read_frame(0, LONG)); Hh, Ww = first.shape[:2]
raw_out = f"{PROJ}/task4/tracked_raw.mp4"; final_out = f"{PROJ}/task4/egg_tracking_output.mp4"
t0 = time.time(); pass1 = run_tracker(frames_from_cap(cap), fps)                       # pass 1: find all confirmed tracks, no drawing
seen_s = pass1.groupby("id").detected.sum() / fps; first_t = pass1.groupby("id").time_s.min()
area_ref = pass1.groupby("id").area.mean()[[i for i in seen_s.index if seen_s[i] >= 3.0]].median()                # typical egg size
good = [i for i in seen_s.index if seen_s[i] >= MIN_SEEN_S and pass1[pass1.id == i].area.mean() >= 0.5 * area_ref]
good.sort(key=lambda i: first_t[i]); keep = {old: new + 1 for new, old in enumerate(good)}
print(f"pass 1: {len(seen_s)} confirmed tracks, {len(good)} kept (seen >= {MIN_SEEN_S}s); dropped short ghost tracks: {len(seen_s) - len(good)}   [{time.time()-t0:.0f}s]")
wr = cv2.VideoWriter(raw_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (Ww, Hh))
trk = run_tracker(frames_from_cap(cap), fps, wr, keep=keep); wr.release()                # pass 2: identical tracking, now drawing only the kept tracks
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_out, "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", "-movflags", "+faststart", final_out], check=True)
trk.to_csv(f"{PROJ}/task4/tracks.csv", index=False)
summ = trk.groupby("id").agg(first_s=("time_s", "min"), last_s=("time_s", "max"), frames=("frame", "count"), detected=("detected", "sum"), mean_area=("area", "mean")).round(2)
print("\nfinal tracks:\n", summ.to_string()); print("\nvideo:", final_out, f"  [{time.time()-t0:.0f}s total]")
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5))
for tid, g in trk.groupby("id"):
    axs[0].plot(g.time_s, g.cx, label=f"egg #{tid}"); axs[1].plot(g.time_s, g.cy, label=f"egg #{tid}")
axs[0].set_title("x position of each tracked egg"); axs[1].set_title("y position of each tracked egg")
for a in axs: a.set_xlabel("time (s)"); a.legend()
plt.tight_layout(); plt.savefig(f"{PROJ}/task4/trajectories.png", dpi=120); plt.show()
os.makedirs("/content/drive/MyDrive/Nosh_Egg_Project_Backup/task4", exist_ok=True)
for fn in ["egg_tracking_output.mp4", "tracks.csv", "trajectories.png"]: shutil.copy(f"{PROJ}/task4/{fn}", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task4/" + fn)
print("copied to Drive: Nosh_Egg_Project_Backup/task4/")
# contact sheet of the output so you can check it without opening the video
cap2 = cv2.VideoCapture(final_out); n2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT)); sel = np.linspace(int(10 * fps), int(42 * fps), 8).astype(int); fig, axs = plt.subplots(2, 4, figsize=(22, 8))
for a, i in zip(axs.ravel(), sel): cap2.set(cv2.CAP_PROP_POS_FRAMES, int(i)); ok, fr = cap2.read(); a.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); a.set_title(f"output frame {i} ({i/fps:.1f}s)"); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ---- re-render only (pass 2) with trail fix: trails are broken across occlusions and only extend where the egg was actually seen ----
def run_tracker(frame_iter, fps, writer=None, keep=None, max_age_s=8.0, confirm_hits=5, cut_thresh=40.0, scene_diff=0.5):
    tracks, next_id, rows, prev_small, ref_hist = [], 1, [], None, None; max_age = int(max_age_s * fps); trails = {}
    for fi, fr in enumerate(frame_iter):
        small = cv2.resize(fr, (64, 36)).astype(np.float32)
        if prev_small is not None and np.abs(small - prev_small).mean() > cut_thresh: tracks, trails = [], {}
        prev_small = small
        dets = detections(video_seg(fr))
        hist = np.bincount((small.mean(2) / 8).astype(int).ravel().clip(0, 31), minlength=32) / small[..., 0].size
        if dets: ref_hist = hist if ref_hist is None else 0.95 * ref_hist + 0.05 * hist
        elif ref_hist is not None and 0.5 * np.abs(hist - ref_hist).sum() > scene_diff: tracks, trails, ref_hist = [], {}, None
        pairs = associate(tracks, dets); mt = {i: j for i, j in pairs}; md = {j for _, j in pairs}
        for i, t in enumerate(tracks):
            if i in mt:
                d = dets[mt[i]]
                if t.misses > int(0.5 * fps): trails.pop(t.id, None)                                   # egg re-appeared after a gap: start a fresh trail
                t.v = 0.5 * t.v + 0.5 * (d["c"] - t.c); vmax = 0.5 * np.sqrt(t.area); t.v = t.v * min(1.0, vmax / (np.linalg.norm(t.v) + 1e-9)); t.c, t.mask, t.area = d["c"], d["mask"], 0.7 * t.area + 0.3 * d["area"]; t.hits += 1; t.misses = 0; t.seen = True
                if t.id is None and t.hits >= confirm_hits: t.id = next_id; next_id += 1
            else:
                t.misses += 1; t.seen = False; t.c = t.c + t.v; t.mask = mask_shift(t.mask, t.v); t.v *= 0.5
        tracks = [t for t in tracks if (t.id is not None and t.misses <= max_age) or (t.id is None and t.misses == 0)]
        tracks += [Track(d["mask"], d["c"], d["area"]) for j, d in enumerate(dets) if j not in md]
        vis = fr.copy(); n_vis = 0
        for t in tracks:
            if t.id is None or (keep is not None and t.id not in keep): continue
            did = t.id if keep is None else keep[t.id]; n_vis += int(t.seen)
            rows.append(dict(frame=fi, time_s=fi / fps, id=did, cx=t.c[0], cy=t.c[1], area=t.area, detected=t.seen))
            col = PALETTE[(did - 1) % len(PALETTE)]
            if t.seen: trails.setdefault(t.id, []).append((int(t.c[0]), int(t.c[1])))                    # only real detections extend the trail
            if t.seen:
                ov = vis.copy(); ov[t.mask] = col; vis = cv2.addWeighted(ov, 0.45, vis, 0.55, 0)
                cs, _ = cv2.findContours(t.mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(vis, cs, -1, col, 2); label = f"egg #{did}"
            elif t.misses <= int(0.5 * fps):
                ys, xs = np.nonzero(t.mask)
                if len(xs): cv2.rectangle(vis, (xs.min(), ys.min()), (xs.max(), ys.max()), col, 1); label = f"egg #{did} (occluded)"
                else: continue
            else: continue
            x0, y0 = int(t.c[0]) - 30, int(t.c[1]) - int(np.sqrt(t.area)) - 8
            cv2.putText(vis, label, (max(x0, 2), max(y0, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 4); cv2.putText(vis, label, (max(x0, 2), max(y0, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
            tr = trails.get(t.id, [])[-40:]
            for a_, b_ in zip(tr[:-1], tr[1:]): cv2.line(vis, a_, b_, col, 2)
        cv2.putText(vis, f"t={fi/fps:5.1f}s  eggs tracked: {n_vis}", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4); cv2.putText(vis, f"t={fi/fps:5.1f}s  eggs tracked: {n_vis}", (10, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        if writer is not None: writer.write(cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return pd.DataFrame(rows)

t0 = time.time(); wr = cv2.VideoWriter(raw_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (Ww, Hh))
trk = run_tracker(frames_from_cap(cap), fps, wr, keep=keep); wr.release()
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_out, "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", "-movflags", "+faststart", final_out], check=True)
trk.to_csv(f"{PROJ}/task4/tracks.csv", index=False)
for fn in ["egg_tracking_output.mp4", "tracks.csv"]: shutil.copy(f"{PROJ}/task4/{fn}", "/content/drive/MyDrive/Nosh_Egg_Project_Backup/task4/" + fn)
print(f"re-rendered in {time.time()-t0:.0f}s, copied to Drive")
cap2 = cv2.VideoCapture(final_out); sel = [int(t * fps) for t in (21.0, 22.0, 23.4, 23.8, 33.5, 34.2)]; fig, axs = plt.subplots(2, 3, figsize=(18, 7))
for a, i in zip(axs.ravel(), sel): cap2.set(cv2.CAP_PROP_POS_FRAMES, i); ok, fr = cap2.read(); a.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); a.set_title(f"{i/fps:.1f}s"); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
import shutil, os, zipfile, glob, subprocess
SRC, DST = PROJ, "/content/drive/MyDrive/Nosh_Egg_Project_Backup"; os.makedirs(DST, exist_ok=True)

# 1) full project to Drive (everything except the SAM checkpoint and the temporary video render)
skip = shutil.ignore_patterns("*.pth", "tracked_raw.mp4", "__pycache__")
for d in ["images", "splits", "task1", "task2", "task3", "task4"]:
    if os.path.isdir(f"{SRC}/{d}"): shutil.copytree(f"{SRC}/{d}", f"{DST}/{d}", dirs_exist_ok=True, ignore=skip)

# 2) small zip with everything needed for the report, saved to Drive and downloaded to your computer
Z = "/content/nosh_deliverables.zip"
want = ["splits/split_manifest.csv", "task1/task1_summary.csv", "task2/experiment_log.csv", "task2/experiment_log.md", "task2/final_config.json",
        "task3/eval_metrics.json", "task3/eval_per_image.csv", "task3/eval_per_egg.csv", "task3/analysis_plots.png", "task3/worst_cases.png", "task3/best_cases.png",
        "task4/tracks.csv", "task4/trajectories.png"]
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as zf:
    for w in want:
        if os.path.exists(f"{SRC}/{w}"): zf.write(f"{SRC}/{w}", w)
    for p in sorted(glob.glob(f"{SRC}/task1/overlays/*.jpg"))[:12]: zf.write(p, "task1/overlays/" + os.path.basename(p))
shutil.copy(Z, DST)
print(f"deliverables.zip: {os.path.getsize(Z)/1e6:.1f} MB")
print(subprocess.run(["du", "-sh", DST], capture_output=True, text=True).stdout)
print(subprocess.run(["ls", DST], capture_output=True, text=True).stdout)
from google.colab import files; files.download(Z)